# XFakeNet – Week 4 on Google Colab

Continues from Weeks 1–3 (reuses everything in your Google Drive). Checks first, then the planned work.

| Section | What it does | Time on T4 |
|---|---|---|
| 0 | Rebuilds the Week 3 models and results **only if they're missing** from Drive | ~40 min if needed |
| A | **Faithfulness test**: do the highlighted words and segments really drive the prediction? + agreement with SHAP | ~5 min |
| B | **Six deception patterns**: Gemini labels ~590 image–headline pairs, a small classifier learns them, and a **hand-check page** downloads for you | ~15 min |
| C | **Integrated pipeline** run on example inputs | ~3 min |
| D | **3 random seeds** for the main models, so the tables show mean ± spread | ~90 min |
| E | *Optional, off by default:* second round of adversarial training | ~50 min |
| F | **Final results tables** + download zip | seconds |
| G | Upload your **hand-check answers** (when you've finished them) | seconds |

**Steps**
1. `Runtime → Change runtime type → T4 GPU`.
2. Key icon (left sidebar) → make sure `GEMINI_API_KEY` is there and **Notebook access** is switched on for this notebook.
3. `Runtime → Run all`, allow Google Drive, keep this tab open.
4. When section B downloads **handcheck.html**, open it in Safari and do the hand-check while section D runs (~1 hour of your time).

If Colab disconnects, `Run all` again: finished parts are skipped.

## 1. GPU, Google Drive, and Week 3 files

In [ ]:
import os, subprocess
gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE – Runtime > Change runtime type > T4 GPU, then Run all again")
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/xfakenet"
os.chdir(WORK)
need = ["runs/baseline/best/config.json", "results/cscore/cscore_results.json", "results/cscore/cscore_pairs.csv",
        "data/augment/paraphrase_train.jsonl", "data/augment/textfooler_train.jsonl", "data/augment/textfooler_test.jsonl"]
w3 = ["runs/xfakenet_v2/xfakenet.pt", "runs/xfakenet_noaug/xfakenet.pt",
      "results/week3/robustness.json", "results/week3/modality_results.json"]
REBUILD_W3 = [f for f in w3 if not os.path.exists(f)]
if REBUILD_W3:
    print("Week 3 outputs missing on Drive:", REBUILD_W3, "-> section 0 will rebuild them (~40 min).")
import shutil
total, used, free = shutil.disk_usage("/content/drive/MyDrive")
print(f"Google Drive: {used/1e9:.1f} GB used of {total/1e9:.1f} GB, {free/1e9:.1f} GB free")
if free < 3e9:
    print("WARNING: less than 3 GB free on Google Drive. New files may silently fail to save. "
          "Empty the Drive Trash and free space before continuing.")
missing = [f for f in need if not os.path.exists(f)]
if missing:
    raise SystemExit(f"Missing Week 1–2 files: {missing}.")
print("Week 1–2 data found in", WORK)

## 2. Write the project code
(These cells just save the scripts into the project folder.)

In [ ]:
import os
for d in ['configs','src/xfakenet','scripts','results/week3','results/week4','results/final','data/deception']:
    os.makedirs(d, exist_ok=True)
print('folders ready')

In [ ]:
%%writefile configs/week1.yaml
seed: 42
paths:
  raw: data/raw
  processed: data/processed
  images: data/images
  runs: runs

liar:
  dir: data/raw/liar            # expects train.tsv / valid.tsv / test.tsv

extra_classes:
  enabled: false      # three-class setup: LIAR only
  source: tshp17
  file: data/raw/tshp17/fulltrain.csv   # columns: label,text (no header)
  test_file: data/raw/tshp17/balancedtest.csv
  use_hoax_trusted: false
  # Articles are long; LIAR statements are ~18 words. Truncate to the first
  # sentences up to this many words so length is not a shortcut for the class.
  max_words: 40
  # Cap per class so Satire/Propaganda stay minority classes (as in the outline).
  max_per_class: 1500

fakenewsnet:
  dir: data/raw/FakeNewsNet/dataset
  sample_size: 1000            # articles to try for images (balanced fake/real)
  min_image_side: 100          # px; smaller images (icons, pixels) are discarded
  workers: 16
  timeout: 10
  use_wayback: true

baseline:
  model: roberta-base
  max_len: 64
  epochs: 4
  batch_size: 32
  lr: 2.0e-5
  warmup_ratio: 0.06
  weight_decay: 0.01
  fp16: true

# ---------------- Week 2 ----------------
augmentation:                  # final training mix (week 3): 70% original / 15% / 15%
  original_share: 70
  textfooler_share: 15
  paraphrase_share: 15

cscore:
  clip_model: openai/clip-vit-base-patch32
  batch_size: 64
  percentile: 95               # alternative threshold: 95th percentile of aligned C-scores (valid)
  out_dir: results/cscore

paraphrase:
  models: [gemini-3.5-flash-lite, gemini-3.8-flash, gemini-3.1-flash-lite, gemini-2.5-flash]
  batch_size: 20               # statements per API call
  temperature: 0.9
  sleep_s: 4                   # pause between calls (free-tier rate limits)
  test_n: 500
  out_train: data/augment/paraphrase_train.jsonl
  out_test: data/augment/paraphrase_test.jsonl

textfooler:
  victim_dir: runs/baseline/best
  embeddings_url: https://github.com/nmrksic/counter-fitting/raw/master/word_vectors/counter-fitted-vectors.txt.zip
  embeddings_path: data/raw/counter-fitted-vectors.txt.zip
  sim_model: sentence-transformers/all-MiniLM-L6-v2
  k: 50                        # synonyms per word (paper: 50)
  sim_threshold: 0.80
  max_change: 0.25             # at most 25% of words replaced
  query_budget: 2000
  test_n: 500
  out_train: data/augment/textfooler_train.jsonl
  out_test: data/augment/textfooler_test.jsonl

model_v1:
  lrs: [1.0e-5, 2.0e-5, 3.0e-5]
  epochs: 5

# ---------------- Week 3 ----------------
xfakenet:                      # RoBERTa + jointly trained token/segment attention
  encoder: roberta-base
  max_len: 64
  max_segments: 8
  att_dim: 256
  epochs: 4
  batch_size: 32
  lr: 2.0e-5                   # encoder
  head_lr: 1.0e-4              # attention layers + classifier
  warmup_ratio: 0.06
  weight_decay: 0.01
  highlight_threshold: 0.65    # outline: tokens above T = 0.65 are highlighted

modality:
  C: 0.1                       # regularisation of the text/image logistic models (389 items)

week3:
  out_dir: results/week3

# ---------------- Week 4 ----------------
week4:
  out_dir: results/week4

deception:
  n_mismatched: 200            # shuffled pairs added to the 389 real ones for labelling
  batch_size: 5                # image-text pairs per Gemini call
  image_side: 512
  sleep_s: 4
  C: 0.5
  min_per_class: 5
  handcheck_n: 100
  labels: data/deception/gemini_labels.jsonl
  features: data/deception/clip_features.npz
  oof: data/deception/classifier_oof.csv
  model: runs/deception_clf.joblib
  handcheck_sample: data/deception/handcheck_sample.csv
  handcheck_results: data/deception/handcheck_results.csv

pipeline:
  modality_model: runs/modality.joblib

explain:                       # faithful explanations (week 4 fix)
  permutations_gpu: 10
  permutations_cpu: 6
  cpu_statements: 400
  chunk: 50

In [ ]:
%%writefile configs/label_mapping.yaml
# XFakeNet five-class label mapping (frozen 2026-10-21, Week 1)
# Every later step (training, augmentation, evaluation) reads this file.
# Change it only with a note in docs/label_mapping.md and a full retrain.

# Three-class setup (changed 20 Oct 2026): Satire/Propaganda dropped because the
# only available source (TSHP-17) taught the model website style, not the class
# (Propaganda F1 0.99 on same-site validation, 0.08 on unseen-site test).
# For the old five-class setup use:
#   classes: [Fabricated, Misleading, Satire, Propaganda, Credible]
#   and set extra_classes.enabled: true in week1.yaml
classes: [Fabricated, Misleading, Credible]   # index order = model label ids

liar:
  # LIAR's six PolitiFact truth ratings -> XFakeNet classes
  # Keys are quoted on purpose: unquoted true/false become YAML booleans.
  "pants-fire":  Fabricated
  "false":       Fabricated
  "barely-true": Misleading
  "half-true":   Misleading
  "mostly-true": Credible
  "true":        Credible

# LIAR has no Satire or Propaganda signal. These two classes come from an
# external labelled source (default: TSHP-17, Rashkin et al. 2017).
tshp17:
  # TSHP-17 integer labels: 1=Satire 2=Hoax 3=Propaganda 4=Trusted
  1: Satire
  3: Propaganda
  # 2 (Hoax) and 4 (Trusted) are NOT used by default: they would add
  # long-article Fabricated/Credible examples whose style differs from LIAR.
  # Set use_hoax_trusted: true in configs/week1.yaml to include them.
  2: Fabricated
  4: Credible

fakenewsnet:
  # Binary only; used for C-score calibration/evaluation, not five-class training.
  "fake": Fabricated
  "real": Credible

In [ ]:
%%writefile src/xfakenet/__init__.py
# package marker

In [ ]:
%%writefile src/xfakenet/common.py
"""Shared helpers: config loading, label mapping, seeding, text cleanup."""
from __future__ import annotations

import json
import random
import re
from pathlib import Path

import yaml

ROOT = Path(__file__).resolve().parents[2]


def load_yaml(path: str | Path) -> dict:
    with open(ROOT / path if not Path(path).is_absolute() else path) as f:
        return yaml.safe_load(f)


def load_config() -> dict:
    return load_yaml("configs/week1.yaml")


def load_mapping() -> dict:
    return load_yaml("configs/label_mapping.yaml")


def class_names() -> list[str]:
    return load_mapping()["classes"]


def label2id() -> dict[str, int]:
    return {c: i for i, c in enumerate(class_names())}


def p(rel: str | Path) -> Path:
    """Resolve a repo-relative path."""
    path = ROOT / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def set_seed(seed: int) -> None:
    random.seed(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


_WS = re.compile(r"\s+")
_SENT = re.compile(r"(?<=[.!?])\s+")


def clean_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.replace("“", '"').replace("”", '"').replace("’", "'")
    return _WS.sub(" ", s).strip()


def truncate_words(text: str, max_words: int) -> str:
    """Keep whole leading sentences up to max_words; hard-cut if the first sentence is longer."""
    text = clean_text(text)
    out, n = [], 0
    for sent in _SENT.split(text):
        w = sent.split()
        if n + len(w) > max_words:
            if not out:
                out.append(" ".join(w[:max_words]))
            break
        out.append(sent)
        n += len(w)
    return " ".join(out)


def write_json(obj, path: str | Path) -> None:
    with open(p(path), "w") as f:
        json.dump(obj, f, indent=2, default=str)

In [ ]:
%%writefile src/xfakenet/textfooler.py
"""TextFooler (Jin et al., AAAI 2020, "Is BERT Really Robust?") without the TextAttack dependency.

Algorithm, as in the paper:
1. Rank words by importance: how much the true-class probability drops when the word is deleted.
2. For each word in that order, take its nearest neighbours in counter-fitted word
   embeddings (synonyms, cos >= 0.5).
3. Keep candidates that preserve the part of speech and keep the sentence meaning
   (sentence-embedding cosine >= threshold to the ORIGINAL text).
4. If a candidate flips the prediction, return the one closest in meaning (success).
   Otherwise keep the candidate that lowers the true-class probability most and continue.

Differences from the paper (documented for the report):
- Sentence similarity uses all-MiniLM-L6-v2 instead of the Universal Sentence Encoder.
- Model queries are batched and constraints are checked lazily (only on candidates
  the model output makes useful), which gives the same result order much faster.
- At most `max_change` (default 25%) of the words may be replaced.
- v2 quality filters (added after inspecting v1 output, where 28% of examples swapped a
  person/party/programme name and so changed the factual claim):
    * proper nouns, numbers and acronyms are never replaced (POS tags NNP/NNPS/CD, any
      capitalised word that is not sentence-initial, all-caps tokens);
    * candidates that are WordNet antonyms of the original word are rejected;
    * candidates on an offensive-word blocklist are rejected.

The class only needs four callables, so it can be unit-tested without a GPU.
"""
from __future__ import annotations

import math
import re
from typing import Callable, Sequence

import numpy as np

# Stopwords are never replaced (subset of the list used by TextFooler).
STOP = set("""a about above across after afterwards again against ain all almost alone along already also
although am among amongst an and another any anyhow anyone anything anyway anywhere are aren aren't around as
at back be became because become becomes becoming been before beforehand behind being below beside besides
between beyond both but by can cannot could couldn couldn't d didn didn't doesn doesn't don don't down due during
either else elsewhere empty enough even ever everyone everything everywhere except first for former formerly from
hadn hadn't hasn hasn't haven haven't he hence her here hereafter hereby herein hereupon hers herself him himself
his how however hundred i if in indeed into is isn isn't it it's its itself just latter latterly least ll may me
meanwhile mightn mightn't mine more moreover most mostly must mustn mustn't my myself namely needn needn't neither
never nevertheless next no nobody none noone nor not nothing now nowhere o of off on once one only onto or other
others otherwise our ours ourselves out over per please s same shan shan't she she's should've shouldn shouldn't
somehow something sometime somewhere such t than that that'll the their theirs them themselves then thence there
thereafter thereby therefore therein thereupon these they this those through throughout thru thus to too toward
towards under unless until up upon used ve was wasn wasn't we were weren weren't what whatever when whence
whenever where whereafter whereas whereby wherein whereupon wherever whether which while whither who whoever whole
whom whose why with within without won won't would wouldn wouldn't y yet you you'd you'll you're you've your yours
yourself yourselves says said""".split())

_AFFIX = re.compile(r"^(\W*)(.*?)(\W*)$")

# Candidates never used as replacements (offensive words that appear in embedding neighbourhoods).
BLOCKLIST = set("""slut sluts whore whores bitch bitches bastard bastards fuck fucking fucked shit shitty
crap cunt dick dicks cock pussy asshole arse arsehole retard retarded faggot fag nigger nigga
negro spic chink kike wetback tranny dyke hooker prostitute rapist""".split())


def split_tok(tok: str) -> tuple[str, str, str]:
    m = _AFFIX.match(tok)
    return m.group(1), m.group(2), m.group(3)


def match_case(src: str, cand: str) -> str:
    if src.isupper() and len(src) > 1:
        return cand.upper()
    if src[:1].isupper():
        return cand[:1].upper() + cand[1:]
    return cand


class TextFooler:
    def __init__(self,
                 predict: Callable[[Sequence[str]], np.ndarray],          # texts -> probs [n, C]
                 neighbours: Callable[[str], list[str]],                   # word -> synonyms
                 similarity: Callable[[str, Sequence[str]], np.ndarray],   # orig, texts -> cos [n]
                 pos_ok: Callable[[list[str], int, str], bool],            # words, idx, cand -> ok
                 protect: Callable[[list[str]], set] | None = None,        # words -> indices never replaced
                 banned: Callable[[str, str], bool] | None = None,         # (orig word, cand) -> reject?
                 sim_threshold: float = 0.80,
                 max_change: float = 0.25,
                 query_budget: int = 2000,
                 max_checks: int = 8):
        self.predict, self.neighbours = predict, neighbours
        self.similarity, self.pos_ok = similarity, pos_ok
        self.sim_threshold, self.max_change = sim_threshold, max_change
        self.query_budget, self.max_checks = query_budget, max_checks
        self.protect, self.banned = protect, banned

    def _ok(self, orig: str, words: list[str], i: int, cand_core: str, text: str) -> tuple[bool, float]:
        if not self.pos_ok(words, i, cand_core):
            return False, 0.0
        s = float(self.similarity(orig, [text])[0])
        return s >= self.sim_threshold, s

    def attack(self, text: str, label: int) -> dict:
        toks = text.split()
        parts = [split_tok(t) for t in toks]
        cores = [c for _, c, _ in parts]
        p0 = self.predict([text])[0]
        q = 1
        if int(np.argmax(p0)) != label:
            return {"status": "skipped", "queries": q}

        protected = set(self.protect(cores)) if self.protect else set()
        elig = [i for i, c in enumerate(cores)
                if c.isalpha() and len(c) >= 3 and c.lower() not in STOP
                and i not in protected
                and not (i > 0 and c[:1].isupper())        # capitalised mid-sentence: a name
                and not c.isupper()]                       # acronyms (CDC, NATO)
        if not elig:
            return {"status": "failed", "queries": q, "reason": "no_eligible_words"}

        # 1. word importance by deletion
        dels = [" ".join(toks[:i] + toks[i + 1:]) for i in elig]
        pd_ = self.predict(dels)
        q += len(dels)
        imp = p0[label] - pd_[:, label]
        flip = pd_.argmax(1) != label
        imp = imp + flip * (pd_.max(1) - p0[pd_.argmax(1)])   # paper's extra term when deletion flips
        order = [elig[k] for k in np.argsort(-imp)]

        cur = list(toks)
        cur_p = float(p0[label])
        changed, max_changes = 0, max(1, math.ceil(self.max_change * len(toks)))

        for i in order:
            if changed >= max_changes or q >= self.query_budget:
                break
            pre, core, post = parts[i]
            syns = [s for s in self.neighbours(core.lower())
                    if s.lower() != core.lower() and s.lower() not in BLOCKLIST
                    and not (self.banned and self.banned(core.lower(), s.lower()))]
            if not syns:
                continue
            cands = [match_case(core, s) for s in syns]
            texts = []
            for c in cands:
                t = list(cur)
                t[i] = pre + c + post
                texts.append(" ".join(t))
            pr = self.predict(texts)
            q += len(texts)
            words_now = [split_tok(t)[1] for t in cur]

            flipped = np.where(pr.argmax(1) != label)[0]
            if len(flipped):
                sims = self.similarity(text, [texts[k] for k in flipped])
                for k_idx in np.argsort(-sims):
                    k = flipped[k_idx]
                    if sims[k_idx] >= self.sim_threshold and self.pos_ok(words_now, i, cands[k]):
                        out = texts[k]
                        return {"status": "success", "text": out, "queries": q,
                                "words_changed": changed + 1,
                                "similarity": round(float(sims[k_idx]), 4),
                                "new_label_id": int(pr[k].argmax()),
                                "orig_conf": round(float(p0[label]), 4),
                                "adv_conf": round(float(pr[k][label]), 4)}

            # no valid flip: keep the best probability-lowering candidate that passes the constraints
            for k in np.argsort(pr[:, label])[: self.max_checks]:
                if pr[k, label] >= cur_p:
                    break
                ok, _ = self._ok(text, words_now, i, cands[k], texts[k])
                if ok:
                    cur[i] = pre + cands[k] + post
                    cur_p = float(pr[k, label])
                    changed += 1
                    break
        return {"status": "failed", "queries": q, "words_changed": changed}


# ---------- real components (used on Colab / the GPU server) ----------

class CounterFitted:
    """Nearest neighbours in counter-fitted vectors (Mrksic et al., 2016), as used by TextFooler."""

    def __init__(self, path: str, k: int = 50, min_cos: float = 0.5):
        words, vecs = [], []
        opener = open
        if path.endswith(".zip"):
            import io
            import zipfile
            z = zipfile.ZipFile(path)
            name = [n for n in z.namelist() if n.endswith(".txt")][0]
            fh = io.TextIOWrapper(z.open(name), encoding="utf-8")
        else:
            fh = opener(path, encoding="utf-8")
        for line in fh:
            w, *v = line.rstrip().split(" ")
            if len(v) < 10:
                continue
            words.append(w)
            vecs.append(np.asarray(v, dtype=np.float32))
        m = np.vstack(vecs)
        self.M = m / np.linalg.norm(m, axis=1, keepdims=True).clip(1e-8)
        self.words, self.idx = words, {w: i for i, w in enumerate(words)}
        self.k, self.min_cos, self.cache = k, min_cos, {}

    def __call__(self, w: str) -> list[str]:
        if w in self.cache:
            return self.cache[w]
        i = self.idx.get(w)
        if i is None:
            self.cache[w] = []
            return []
        sims = self.M @ self.M[i]
        kk = min(self.k + 1, len(sims) - 1)
        top = np.argpartition(-sims, kk)[: kk + 1]
        top = top[np.argsort(-sims[top])]
        res = [self.words[j] for j in top if j != i and sims[j] >= self.min_cos][: self.k]
        self.cache[w] = res
        return res


class WordNetSyn:
    """Fallback synonym source if the counter-fitted vectors cannot be downloaded."""

    def __init__(self, k: int = 50):
        from nltk.corpus import wordnet as wn
        self.wn, self.k, self.cache = wn, k, {}

    def __call__(self, w: str) -> list[str]:
        if w not in self.cache:
            out = []
            for s in self.wn.synsets(w):
                for l in s.lemma_names():
                    l = l.replace("_", " ")
                    if " " not in l and l.lower() != w and l not in out:
                        out.append(l)
            self.cache[w] = out[: self.k]
        return self.cache[w]


def make_pos_ok(window: int = 3):
    import nltk

    def coarse(tag: str) -> str:
        return tag[0] if tag[0] in "NVJR" else tag

    def pos_ok(words: list[str], i: int, cand: str) -> bool:
        lo, hi = max(0, i - window), min(len(words), i + window + 1)
        a = [w if w else "," for w in words[lo:hi]]
        b = list(a)
        b[i - lo] = cand
        ta, tb = nltk.pos_tag(a), nltk.pos_tag(b)
        return coarse(ta[i - lo][1]) == coarse(tb[i - lo][1])

    return pos_ok


def make_protect():
    """Indices of proper nouns and numbers (never replaced)."""
    import nltk

    def protect(words: list[str]) -> set:
        tags = nltk.pos_tag([w if w else "," for w in words])
        return {i for i, (_, t) in enumerate(tags) if t in ("NNP", "NNPS", "CD")}

    return protect


def make_banned():
    """Reject WordNet antonyms of the original word (e.g. died -> lives)."""
    from nltk.corpus import wordnet as wn
    cache: dict[str, set] = {}

    def antonyms(w: str) -> set:
        if w not in cache:
            out = set()
            for syn in wn.synsets(w):
                for lem in syn.lemmas():
                    for a in lem.antonyms():
                        out.add(a.name().lower())
                        for rel in a.derivationally_related_forms():
                            out.add(rel.name().lower())
            cache[w] = out
        return cache[w]

    def banned(orig: str, cand: str) -> bool:
        if cand in antonyms(orig) or orig in antonyms(cand):
            return True
        # also compare base forms (died/die vs lives/live)
        base_o, base_c = wn.morphy(orig) or orig, wn.morphy(cand) or cand
        return base_c in antonyms(base_o) or base_o in antonyms(base_c)

    return banned


def make_similarity(model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer(model_name)
    cache: dict[str, np.ndarray] = {}

    def sim(orig: str, texts: Sequence[str]) -> np.ndarray:
        if orig not in cache:
            if len(cache) > 50:
                cache.clear()
            cache[orig] = st.encode([orig], normalize_embeddings=True)[0]
        e = st.encode(list(texts), normalize_embeddings=True, batch_size=64)
        return e @ cache[orig]

    return sim


def make_predict(model_dir: str, max_len: int = 64, batch_size: int = 64):
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(dev).eval()

    @torch.no_grad()
    def predict(texts: Sequence[str]) -> np.ndarray:
        out = []
        for s in range(0, len(texts), batch_size):
            enc = tok(list(texts[s:s + batch_size]), truncation=True, max_length=max_len,
                      padding=True, return_tensors="pt").to(dev)
            with torch.autocast(device_type="cuda", enabled=dev == "cuda"):
                logits = model(**enc).logits
            out.append(torch.softmax(logits.float(), -1).cpu().numpy())
        return np.concatenate(out)

    return predict

In [ ]:
%%writefile src/xfakenet/segments.py
"""Split a statement into segments (sentences and clauses) for the sentence-attention layer.

About 81% of LIAR statements are a single sentence, so splitting only at sentence
boundaries would give the attention layer nothing to choose between. We also split
at ; : commas and dashes, then merge pieces shorter than `min_words` into their
neighbour so a segment is always a meaningful clause.
"""
from __future__ import annotations

import re

_SPLIT = re.compile(r"(?<=[.!?;:])\s+|,\s+|\s+--?\s+")


def segment_spans(text: str, max_segments: int = 8, min_words: int = 3) -> list[tuple[int, int]]:
    """Return character spans (start, end) of the segments of `text`."""
    spans, start = [], 0
    for m in _SPLIT.finditer(text):
        if m.start() > start:
            spans.append((start, m.start()))
        start = m.end()
    if start < len(text):
        spans.append((start, len(text)))
    if not spans:
        return [(0, len(text))]

    words = lambda s: len(text[s[0]:s[1]].split())  # noqa: E731
    merged: list[tuple[int, int]] = []
    for s in spans:
        if merged and (words(s) < min_words or words(merged[-1]) < min_words):
            merged[-1] = (merged[-1][0], s[1])
        else:
            merged.append(s)
    if len(merged) > max_segments:
        merged = merged[: max_segments - 1] + [(merged[max_segments - 1][0], merged[-1][1])]
    return merged


def segments(text: str, max_segments: int = 8, min_words: int = 3) -> list[str]:
    return [text[a:b] for a, b in segment_spans(text, max_segments, min_words)]

In [ ]:
%%writefile src/xfakenet/explain_utils.py
"""Word- and segment-level explanation from attention weights (no torch needed)."""
from __future__ import annotations

from bisect import bisect_right

import numpy as np


def word_explanation(text, offsets, seg_ids, tok_att, seg_att, spans, probs, classes, thr) -> dict:
    """Turn token/segment attention into word-level and segment-level explanations (pure numpy)."""
    words, w_imp, pos = [], [], 0
    for w in text.split():
        a = text.index(w, pos)
        words.append((a, a + len(w), w))
        w_imp.append(0.0)
        pos = a + len(w)
    starts = [a for a, _, _ in words]
    for j, (a, z) in enumerate(offsets):
        if z > a and seg_ids[j] >= 0 and words:
            k = max(0, bisect_right(starts, a) - 1)
            w_imp[k] += float(tok_att[j])      # a word's importance = sum over its sub-word tokens
    m = max(w_imp) if w_imp and max(w_imp) > 0 else 1.0
    norm = [x / m for x in w_imp]
    segs = [{"text": text[a:z], "weight": round(float(seg_att[k]), 4)}
            for k, (a, z) in enumerate(spans) if k < len(seg_att)]
    top = int(np.argmax([s["weight"] for s in segs])) if segs else 0
    return {
        "text": text,
        "pred": classes[int(np.argmax(probs))],
        "probs": {c: round(float(p_), 4) for c, p_ in zip(classes, probs)},
        "words": [{"w": w, "weight": round(x, 4)} for (_, _, w), x in zip(words, norm)],
        "highlighted": [w for (_, _, w), x in zip(words, norm) if x >= thr],
        "segments": segs,
        "top_segment": segs[top]["text"] if segs else text,
    }

In [ ]:
%%writefile src/xfakenet/model.py
"""XFakeNet classifier: RoBERTa + jointly trained token- and segment-attention (Module 1 + 3).

    tokens --RoBERTa--> h_t
    token attention (within each segment):   a_t = softmax_t( v_t . tanh(W_t h_t) )
    segment vector:                          s_k = sum_t a_t h_t
    segment attention:                       alpha_k = softmax_k( v_s . tanh(W_s s_k) )
    statement vector:                        d = sum_k alpha_k s_k
    prediction:                              softmax(W_c d)

The classifier only sees d, so the attention weights are part of the decision
itself (explanation produced in the forward pass), not computed afterwards.
Token importance shown to the user = a_t * alpha_(segment of t).
"""
from __future__ import annotations

import json
from bisect import bisect_right
from pathlib import Path
from typing import Sequence

import numpy as np
import torch
from torch import nn

from .explain_utils import word_explanation
from .segments import segment_spans


class XFakeNetModel(nn.Module):
    def __init__(self, encoder: nn.Module, num_labels: int, max_segments: int = 8,
                 att_dim: int = 256, dropout: float = 0.1):
        super().__init__()
        self.encoder = encoder
        d = encoder.config.hidden_size
        self.K = max_segments
        self.tok_proj, self.tok_v = nn.Linear(d, att_dim), nn.Linear(att_dim, 1, bias=False)
        self.seg_proj, self.seg_v = nn.Linear(d, att_dim), nn.Linear(att_dim, 1, bias=False)
        self.drop = nn.Dropout(dropout)
        self.cls = nn.Linear(d, num_labels)

    def forward(self, input_ids, attention_mask, seg_ids):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state  # B L D
        b, l, _ = h.shape
        e = self.tok_v(torch.tanh(self.tok_proj(h))).squeeze(-1).float()                       # B L
        ks = torch.arange(self.K, device=h.device)
        member = seg_ids.unsqueeze(1) == ks.view(1, -1, 1)                                       # B K L
        scores = e.unsqueeze(1).expand(b, self.K, l).masked_fill(~member, -1e4)
        a = torch.softmax(scores, -1) * member                                                   # B K L (empty segment -> 0)
        s = torch.bmm(a.to(h.dtype), h)                                                          # B K D
        valid = member.any(-1)                                                                   # B K
        u = self.seg_v(torch.tanh(self.seg_proj(s))).squeeze(-1).float().masked_fill(~valid, -1e4)
        alpha = torch.softmax(u, -1) * valid                                                     # B K
        doc = torch.bmm(alpha.unsqueeze(1).to(s.dtype), s).squeeze(1)                            # B D
        logits = self.cls(self.drop(doc)).float()
        token_imp = (a * alpha.unsqueeze(-1)).sum(1)                                             # B L
        return {"logits": logits, "token_att": token_imp, "seg_att": alpha}


def encode(tok, texts: Sequence[str], max_len: int, max_segments: int, device=None):
    """Tokenise and assign every token to a segment (special/pad tokens get -1)."""
    enc = tok(list(texts), truncation=True, max_length=max_len, padding=True,
              return_offsets_mapping=True, return_tensors="pt")
    offsets = enc.pop("offset_mapping")
    seg_ids = torch.full(enc["input_ids"].shape, -1, dtype=torch.long)
    spans_all = []
    for i, text in enumerate(texts):
        spans = segment_spans(text, max_segments)
        starts = [a for a, _ in spans]
        spans_all.append(spans)
        for j, (a, z) in enumerate(offsets[i].tolist()):
            if z > a:
                seg_ids[i, j] = max(0, bisect_right(starts, a) - 1)
    enc["seg_ids"] = seg_ids
    if device is not None:
        enc = {k: v.to(device) for k, v in enc.items()}
    return enc, offsets, spans_all


def build(encoder_name: str, num_labels: int, max_segments: int, att_dim: int) -> XFakeNetModel:
    from transformers import AutoModel
    return XFakeNetModel(AutoModel.from_pretrained(encoder_name), num_labels, max_segments, att_dim)


def save(model: XFakeNetModel, tok, out_dir: str | Path, meta: dict) -> None:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), out / "xfakenet.pt")
    model.encoder.config.save_pretrained(out / "encoder_config")
    tok.save_pretrained(out)
    json.dump(meta, open(out / "xfakenet_meta.json", "w"), indent=2)


def load(model_dir: str | Path, device: str = "cpu"):
    from transformers import AutoConfig, AutoModel, AutoTokenizer
    d = Path(model_dir)
    meta = json.load(open(d / "xfakenet_meta.json"))
    enc = AutoModel.from_config(AutoConfig.from_pretrained(d / "encoder_config"))
    model = XFakeNetModel(enc, len(meta["classes"]), meta["max_segments"], meta["att_dim"])
    model.load_state_dict(torch.load(d / "xfakenet.pt", map_location="cpu"))
    return model.to(device).eval(), AutoTokenizer.from_pretrained(d), meta


class Predictor:
    """Batch prediction + explanations for a saved XFakeNet model."""

    def __init__(self, model_dir: str | Path, batch_size: int = 64, device: str | None = None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model, self.tok, self.meta = load(model_dir, self.device)
        self.bs = batch_size

    @torch.no_grad()
    def _run(self, texts: Sequence[str]):
        for s in range(0, len(texts), self.bs):
            batch = list(texts[s:s + self.bs])
            enc, offsets, spans = encode(self.tok, batch, self.meta["max_len"], self.meta["max_segments"],
                                         self.device)
            with torch.autocast(device_type="cuda", enabled=self.device == "cuda"):
                out = self.model(**enc)
            yield batch, enc, offsets, spans, out

    def predict_proba(self, texts: Sequence[str]) -> np.ndarray:
        res = [torch.softmax(o["logits"], -1).cpu().numpy() for *_, o in self._run(texts)]
        return np.concatenate(res) if res else np.zeros((0, len(self.meta["classes"])))

    def explain(self, texts: Sequence[str], threshold: float | None = None) -> list[dict]:
        thr = self.meta.get("highlight_threshold", 0.65) if threshold is None else threshold
        out = []
        for batch, enc, offsets, spans, o in self._run(texts):
            probs = torch.softmax(o["logits"], -1).cpu().numpy()
            tok_att = o["token_att"].float().cpu().numpy()
            seg_att = o["seg_att"].float().cpu().numpy()
            seg_ids = enc["seg_ids"].cpu().numpy()
            offs = offsets.tolist()
            for i, text in enumerate(batch):
                out.append(word_explanation(text, offs[i], seg_ids[i], tok_att[i], seg_att[i], spans[i],
                                            probs[i], self.meta["classes"], thr))
        return out

In [ ]:
%%writefile src/xfakenet/clip_utils.py
"""CLIP ViT-B/32 helpers shared by the C-score, modality flag, deception classifier and pipeline."""
from __future__ import annotations

import numpy as np


class Clip:
    def __init__(self, name: str = "openai/clip-vit-base-patch32", device: str | None = None):
        import torch
        from transformers import CLIPModel, CLIPProcessor
        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = CLIPModel.from_pretrained(name).to(self.device).eval()
        self.proc = CLIPProcessor.from_pretrained(name)

    def embed(self, titles: list[str], images: list, batch_size: int = 64):
        """Normalised (text, image) embeddings. `images` are PIL images or file paths."""
        from PIL import Image
        T, I = [], []
        for s in range(0, len(titles), batch_size):
            ims = [Image.open(x).convert("RGB") if isinstance(x, (str, bytes)) or hasattr(x, "__fspath__")
                   else x.convert("RGB") for x in images[s:s + batch_size]]
            inp = self.proc(text=list(titles[s:s + batch_size]), images=ims, return_tensors="pt",
                            padding=True, truncation=True, max_length=77).to(self.device)
            with self.torch.no_grad():
                out = self.model(**inp)
            T.append(self.torch.nn.functional.normalize(out.text_embeds, dim=-1).cpu().numpy())
            I.append(self.torch.nn.functional.normalize(out.image_embeds, dim=-1).cpu().numpy())
        return np.concatenate(T), np.concatenate(I)


def pair_features(T: np.ndarray, I: np.ndarray) -> np.ndarray:
    """Features for the deception-pattern classifier: text, image, their product, and the C-score."""
    c = 1.0 - (T * I).sum(1, keepdims=True)
    return np.hstack([T, I, T * I, c])

In [ ]:
%%writefile src/xfakenet/pipeline.py
"""The integrated XFakeNet pipeline: one call returns class, explanations, modality flag and pattern.

    pipe = XFakeNetPipeline.load()
    out = pipe("Headline or statement text", image="path/or/PIL.Image")   # image optional

Output keys:
  label, probs                      classifier decision (Fabricated / Misleading / Credible)
  highlighted, word_scores,         Module 1 explanations. Highlights and the top segment come from
  segments, top_segment             SHAP (faithful); the built-in attention versions are kept as
  attention_highlighted,            attention_highlighted / attention_top_segment for comparison.
  attention_top_segment
  modality_flag                     TEXT / IMAGE / CROSS-MODAL (TEXT when no image is given)
  cscore, cscore_threshold          Module 2 contradiction score (only with an image)
  deception_pattern, pattern_probs  one of six patterns (only when CROSS-MODAL)
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np

from .common import load_config, p


class XFakeNetPipeline:
    def __init__(self, model_dir, cscore_threshold, modality_path=None, deception_path=None,
                 clip_name="openai/clip-vit-base-patch32", shap_permutations: int = 10, threshold: float = 0.65):
        from .model import Predictor
        self.pred = Predictor(model_dir)
        self.thr = float(cscore_threshold)
        self.m, self.hl_thr = shap_permutations, threshold
        self.clip_name = clip_name
        self._clip = None
        self.modality = self._load(modality_path)
        self.deception = self._load(deception_path)

    @staticmethod
    def _load(path):
        if path and Path(path).exists():
            import joblib
            return joblib.load(path)
        return None

    @classmethod
    def load(cls, model_dir: str = "runs/xfakenet_v2"):
        cfg = load_config()
        thr = json.load(open(p(f"{cfg['cscore']['out_dir']}/cscore_results.json")))["chosen_threshold"]
        return cls(p(model_dir), thr, p(cfg["pipeline"]["modality_model"]), p(cfg["deception"]["model"]),
                   cfg["cscore"]["clip_model"], cfg["explain"]["permutations_gpu"],
                   cfg["xfakenet"]["highlight_threshold"])

    @property
    def clip(self):
        if self._clip is None:
            from .clip_utils import Clip
            self._clip = Clip(self.clip_name)
        return self._clip

    def __call__(self, text: str, image=None) -> dict:
        from .attribution import explain_batch, highlight
        e = self.pred.explain([text])[0]
        words, segs = [w["w"] for w in e["words"]], [s["text"] for s in e["segments"]]
        a = explain_batch(self.pred.predict_proba, [{"words": words, "segments": segs, "pred": e["pred"]}],
                          self.pred.meta["classes"], m=self.m, methods=("shap", "segment"))[0]
        shap = a.get("shap", [0.0] * len(words))
        seg_sc = a.get("segment_shap") or [s["weight"] for s in e["segments"]]
        top_seg = segs[int(np.argmax(seg_sc))] if segs else text
        out = {"label": e["pred"], "probs": e["probs"],
               "highlighted": highlight(words, shap, self.hl_thr),
               "word_scores": [{"w": w, "shap": round(float(x), 4)} for w, x in zip(words, shap)],
               "segments": [{"text": s, "shap": round(float(x), 4)} for s, x in zip(segs, seg_sc)],
               "top_segment": top_seg,
               "attention_highlighted": e["highlighted"], "attention_top_segment": e["top_segment"],
               "modality_flag": "TEXT", "cscore": None, "cscore_threshold": self.thr,
               "deception_pattern": None, "pattern_probs": None}
        if image is None:
            return out
        from .clip_utils import pair_features
        T, I = self.clip.embed([text], [image])
        c = float(1.0 - (T * I).sum())
        out["cscore"] = round(c, 4)
        if c > self.thr:
            out["modality_flag"] = "CROSS-MODAL"
            if self.deception is not None:
                pr = self.deception["model"].predict_proba(pair_features(T, I))[0]
                cls = list(self.deception["model"].classes_)
                out["pattern_probs"] = {k: round(float(v), 4) for k, v in zip(cls, pr)}
                out["deception_pattern"] = cls[int(np.argmax(pr))]
        elif self.modality is not None:
            m = self.modality
            zt = float(m["text"].decision_function(T)[0])
            zi = float(m["image"].decision_function(I)[0])
            z = m["fusion_scaler"].transform([[zt, zi]])[0]
            ct, ci = m["fusion_coef"][0] * z[0], m["fusion_coef"][1] * z[1]
            out["modality_flag"] = "TEXT" if abs(ct) >= abs(ci) else "IMAGE"
            out["modality_contrib"] = {"text": round(float(ct), 4), "image": round(float(ci), 4)}
        return out

In [ ]:
%%writefile scripts/06_train_baseline.py
"""Week-1 baseline: RoBERTa-base, five classes, class-weighted cross-entropy.

This is the "fine-tuned RoBERTa without the proposed modules" baseline from the
evaluation plan. It is also the victim model for TextFooler generation.

Input : data/processed/corpus_{train,valid,test}.csv, corpus_stats.json
Output: runs/baseline/best/            (model + tokenizer, chosen by valid macro-F1)
        runs/baseline/metrics.json     (valid + test: acc, macro-F1, per-class, per-source)
        runs/baseline/confusion_test.csv

Usage : python scripts/06_train_baseline.py [--epochs N] [--smoke]
"""
from __future__ import annotations

import argparse
import json
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch import nn
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import class_names, load_config, p, set_seed, write_json  # noqa: E402


class TextDS(torch.utils.data.Dataset):
    def __init__(self, df: pd.DataFrame, tok, max_len: int):
        self.enc = tok(df["text"].tolist(), truncation=True, max_length=max_len)
        self.y = df["label_id"].astype(int).tolist()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.y[i]
        return item


class WeightedTrainer(Trainer):
    def __init__(self, *a, class_weights: torch.Tensor, **kw):
        super().__init__(*a, **kw)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = nn.functional.cross_entropy(out.logits, labels,
                                           weight=self.class_weights.to(out.logits.device))
        return (loss, out) if return_outputs else loss


def metrics_fn(eval_pred):
    logits, labels = eval_pred
    pred = logits.argmax(-1)
    present = sorted(set(labels.tolist()))
    return {"accuracy": accuracy_score(labels, pred),
            "macro_f1": f1_score(labels, pred, labels=present, average="macro", zero_division=0)}


def report(df: pd.DataFrame, pred: np.ndarray, classes: list[str]) -> dict:
    y = df["label_id"].to_numpy()
    present = sorted(set(y.tolist()))
    r = {
        "n": len(df),
        "accuracy": round(accuracy_score(y, pred), 4),
        "macro_f1": round(f1_score(y, pred, labels=present, average="macro", zero_division=0), 4),
        "classes_in_eval": [classes[i] for i in present],
        "per_class": classification_report(y, pred, labels=list(range(len(classes))),
                                           target_names=classes, output_dict=True, zero_division=0),
        "per_source": {},
    }
    for src, idx in df.groupby("source").groups.items():
        ys, ps = y[df.index.get_indexer(idx)], pred[df.index.get_indexer(idx)]
        pr = sorted(set(ys.tolist()))
        r["per_source"][src] = {"n": len(ys), "accuracy": round(accuracy_score(ys, ps), 4),
                                "macro_f1": round(f1_score(ys, ps, labels=pr, average="macro",
                                                           zero_division=0), 4)}
    return r


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=float, default=None)
    ap.add_argument("--smoke", action="store_true", help="tiny model + 200 rows, CPU-friendly check")
    ap.add_argument("--lr", type=float, default=None)
    ap.add_argument("--run-name", default="baseline", help="output folder under runs/")
    ap.add_argument("--seed", type=int, default=None)
    ap.add_argument("--out-dir", default=None, help="absolute output folder (e.g. Colab local disk instead of Drive)")
    args = ap.parse_args()

    cfg = load_config()
    if args.seed is not None:
        cfg["seed"] = args.seed
    bc = cfg["baseline"]
    set_seed(cfg["seed"])
    classes = class_names()
    d = cfg["paths"]["processed"]
    out_dir = Path(args.out_dir) if args.out_dir else p(f"{cfg['paths']['runs']}/{args.run_name}")
    out_dir.mkdir(parents=True, exist_ok=True)

    train, valid, test = (pd.read_csv(p(f"{d}/corpus_{s}.csv")) for s in ["train", "valid", "test"])
    stats = json.load(open(p(f"{d}/corpus_stats.json")))
    weights = torch.tensor([stats["class_weights"][c] for c in classes], dtype=torch.float)

    model_name = bc["model"]
    if args.smoke:
        model_name = "hf-internal-testing/tiny-random-RobertaForSequenceClassification"
        train, valid, test = train.head(200), valid.head(100), test.head(100)

    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(classes),
        id2label=dict(enumerate(classes)), label2id={c: i for i, c in enumerate(classes)},
        ignore_mismatched_sizes=args.smoke)

    ds = {k: TextDS(v, tok, bc["max_len"]) for k, v in
          [("train", train), ("valid", valid), ("test", test)]}

    targs = TrainingArguments(
        output_dir=str(out_dir / "ckpt"),
        num_train_epochs=args.epochs or bc["epochs"],
        per_device_train_batch_size=bc["batch_size"],
        per_device_eval_batch_size=bc["batch_size"] * 2,
        learning_rate=args.lr or bc["lr"],
        # warmup as an integer step count: works on transformers 4.x and 5.x
        # (5.x removed warmup_ratio)
        warmup_steps=int(math.ceil(len(train) / bc["batch_size"])
                         * (args.epochs or bc["epochs"]) * bc["warmup_ratio"]),
        weight_decay=bc["weight_decay"],
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=bc["fp16"] and torch.cuda.is_available(),
        logging_steps=50,
        report_to="none",
        seed=cfg["seed"],
    )
    trainer = WeightedTrainer(model=model, args=targs, train_dataset=ds["train"],
                              eval_dataset=ds["valid"], data_collator=DataCollatorWithPadding(tok),
                              compute_metrics=metrics_fn, class_weights=weights)
    trainer.train()
    trainer.save_model(str(out_dir / "best"))
    tok.save_pretrained(str(out_dir / "best"))

    results = {"model": model_name, "class_weights": stats["class_weights"], "config": bc,
               "run_name": args.run_name, "seed": cfg["seed"], "lr": args.lr or bc["lr"], "epochs": args.epochs or bc["epochs"]}
    for name, df in [("valid", valid), ("test", test)]:
        pred = trainer.predict(ds[name]).predictions.argmax(-1)
        results[name] = report(df.reset_index(drop=True), pred, classes)
        print(f"[{args.run_name}] {name}: acc={results[name]['accuracy']} macro_f1={results[name]['macro_f1']} "
              f"per_source={results[name]['per_source']}")
        if name == "test":
            cm = confusion_matrix(df["label_id"], pred, labels=list(range(len(classes))))
            pd.DataFrame(cm, index=classes, columns=classes).to_csv(out_dir / "confusion_test.csv")
            pd.DataFrame({"id": df["id"], "text": df["text"], "label": df["label"],
                          "pred": [classes[i] for i in pred]}).to_csv(out_dir / "test_predictions.csv",
                                                                       index=False)
    write_json(results, out_dir / "metrics.json")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/07_textfooler_generate.py
"""Generate TextFooler adversarial examples against the week-1 baseline.

Uses src/xfakenet/textfooler.py (the Jin et al. algorithm, no TextAttack needed).

--split train : attacks TRAIN statements until the number of SUCCESSFUL examples
                reaches 15% of the final corpus (70/15/15), with per-class quotas.
                These are the augmentation data for week 3 (gold label kept).
--split test  : attacks a fixed stratified sample of the TEST split. The attack
                success rate is the robustness number for week 4. Never trained on.

Resumable: ids already in the output file are skipped and existing successes count.

Usage: python scripts/07_textfooler_generate.py --split train
       python scripts/07_textfooler_generate.py --split test
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time
import urllib.request
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import load_config, p, set_seed  # noqa: E402
from xfakenet import textfooler as tf  # noqa: E402

TF_VERSION = 2   # v2: names/numbers protected, antonyms + offensive words blocked


def synonym_source(cfg: dict):
    import zipfile
    tc = cfg["textfooler"]
    path = p(tc["embeddings_path"])
    urls = [tc["embeddings_url"],
            "https://media.githubusercontent.com/media/nmrksic/counter-fitting/master/word_vectors/counter-fitted-vectors.txt.zip"]
    for url in urls:
        if path.exists() and zipfile.is_zipfile(path):
            break
        if path.exists():
            os.remove(path)
        print(f"[textfooler] downloading counter-fitted vectors from {url} ...", flush=True)
        try:
            urllib.request.urlretrieve(url, path)
        except Exception as e:
            print(f"[textfooler] download failed ({e})", flush=True)
    if path.exists() and not zipfile.is_zipfile(path):
        os.remove(path)
        print("[textfooler] download was not a valid zip; using WordNet synonyms instead", flush=True)
    if path.exists():
        print("[textfooler] synonyms: counter-fitted vectors", flush=True)
        return tf.CounterFitted(str(path), k=tc["k"], min_cos=0.5), "counter-fitted"
    import nltk
    nltk.download("wordnet", quiet=True)
    return tf.WordNetSyn(k=tc["k"]), "wordnet"


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--split", default="train", choices=["train", "test"])
    ap.add_argument("--n", type=int, default=None)
    ap.add_argument("--victim", default=None)
    ap.add_argument("--victim-kind", default="hf", choices=["hf", "xfakenet"],
                    help="hf = Hugging Face classifier (baseline); xfakenet = XFakeNet model (week 3)")
    ap.add_argument("--out", default=None, help="override output file")
    args = ap.parse_args()

    cfg = load_config()
    tc = cfg["textfooler"]
    set_seed(cfg["seed"])

    import nltk
    for r in ["averaged_perceptron_tagger_eng", "averaged_perceptron_tagger", "wordnet"]:
        nltk.download(r, quiet=True)

    syn, syn_name = synonym_source(cfg)
    victim = str(p(args.victim or tc["victim_dir"]))
    if args.victim_kind == "xfakenet":
        from xfakenet.model import Predictor
        predict_fn = Predictor(victim).predict_proba
    else:
        predict_fn = tf.make_predict(victim, cfg["baseline"]["max_len"])
    attacker = tf.TextFooler(
        predict=predict_fn,
        neighbours=syn, similarity=tf.make_similarity(tc["sim_model"]), pos_ok=tf.make_pos_ok(),
        protect=tf.make_protect(), banned=tf.make_banned(),
        sim_threshold=tc["sim_threshold"], max_change=tc["max_change"], query_budget=tc["query_budget"])

    df = pd.read_csv(p(f"{cfg['paths']['processed']}/corpus_{args.split}.csv"))
    df["id"] = df["id"].astype(str)
    out = p(args.out) if args.out else p(tc["out_train"] if args.split == "train" else tc["out_test"])
    prev = [json.loads(line) for line in open(out)] if out.exists() else []
    if prev and prev[0].get("tf_version") != TF_VERSION:
        old = out.with_name(out.stem + ".v1_unfiltered.jsonl")
        out.rename(old)
        print(f"[textfooler] archived old unfiltered output to {old.name}; regenerating", flush=True)
        prev = []
    done = {r["id"] for r in prev}

    if args.split == "train":
        a = cfg["augmentation"]
        target = args.n or round(len(df) * a["textfooler_share"] / a["original_share"])
        frac = df["label"].value_counts(normalize=True)
        quota = {c: round(target * f) for c, f in frac.items()}
        have = pd.Series([r["label"] for r in prev if r["status"] == "success"]).value_counts().to_dict()
        order = df.sample(frac=1.0, random_state=cfg["seed"])        # whole train, shuffled
    else:
        target = args.n or tc["test_n"]
        frac = df["label"].value_counts(normalize=True)
        order = pd.concat([g.sample(min(len(g), round(target * frac[c])), random_state=cfg["seed"])
                           for c, g in df.groupby("label")]).sample(frac=1.0, random_state=cfg["seed"])
        quota, have = None, {}
    todo = order[~order["id"].isin(done)]
    print(f"[textfooler] victim={victim} split={args.split} synonyms={syn_name} target={target} "
          f"already_done={len(done)} successes_so_far={sum(have.values())}", flush=True)

    t0, n, counts = time.time(), 0, {}
    with open(out, "a") as f:
        for row in todo.itertuples(index=False):
            if quota is not None:
                if all(have.get(c, 0) >= q for c, q in quota.items()):
                    break
                if have.get(row.label, 0) >= quota[row.label]:
                    continue
            try:
                res = attacker.attack(row.text, int(row.label_id))
            except Exception as e:  # keep the long job alive
                res = {"status": "error", "error": repr(e)[:200]}
            rec = {"tf_version": TF_VERSION, "id": row.id, "label": row.label, "label_id": int(row.label_id),
                   "source": row.source, "original": row.text, "synonyms": syn_name, **res}
            f.write(json.dumps(rec) + "\n")
            f.flush()
            counts[res["status"]] = counts.get(res["status"], 0) + 1
            if res["status"] == "success":
                have[row.label] = have.get(row.label, 0) + 1
            n += 1
            if n % 50 == 0:
                rate = n / (time.time() - t0)
                print(f"[textfooler] {n} attacked | {counts} | successes {sum(have.values())}/{target} "
                      f"| {rate:.1f}/s", flush=True)
    print(f"[textfooler] finished {counts} | successes by class {have}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/08_clip_cscore.py
"""Module 2: CLIP contradiction score (C-score) and threshold calibration.

C-score = 1 - cos(E_text, E_image), CLIP ViT-B/32, headline vs article image.

There are no human labels for "this image contradicts this headline", so the
threshold is calibrated the standard way: real headline+image pairs are the
ALIGNED class, and the same headlines paired with another article's image
(shuffled within the same site, so politics images stay with politics) are the
MISMATCHED class. The threshold is fitted on a validation split and reported
on a held-out test split.

Input : data/processed/fnn_pairs.csv, data/images/
Output: results/cscore/cscore_pairs.csv     (every aligned + mismatched pair with its C-score)
        results/cscore/cscore_results.json  (thresholds, AUC, accuracy, fake-vs-real check)
        results/cscore/cscore_hist.png
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import load_config, p, set_seed, write_json  # noqa: E402


def derange_within(groups: pd.Series, rng: np.random.Generator) -> np.ndarray:
    """For each row return the index of a DIFFERENT row from the same group."""
    idx = np.arange(len(groups))
    out = np.empty_like(idx)
    for _, g in pd.Series(idx).groupby(groups.values):
        g = g.to_numpy()
        if len(g) < 2:
            out[g] = g  # cannot mismatch a singleton; dropped later
            continue
        out[g] = _derange(g, rng)
    return out


def _derange(g: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    perm = rng.permutation(len(g))
    shifted = np.roll(perm, 1)              # cyclic shift of a random order: no fixed points
    res = np.empty_like(g)
    res[perm] = g[shifted]
    return res


def youden_threshold(scores: np.ndarray, labels: np.ndarray) -> float:
    """Threshold maximising TPR - FPR (labels: 1 = mismatched)."""
    from sklearn.metrics import roc_curve
    fpr, tpr, thr = roc_curve(labels, scores)
    j = tpr - fpr
    t = thr[int(np.argmax(j))]
    return float(t if np.isfinite(t) else np.median(scores))


def eval_at(scores, labels, t) -> dict:
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
    pred = (scores > t).astype(int)
    return {"threshold": round(float(t), 4),
            "accuracy": round(accuracy_score(labels, pred), 4),
            "f1_mismatch": round(f1_score(labels, pred, zero_division=0), 4),
            "precision": round(precision_score(labels, pred, zero_division=0), 4),
            "recall": round(recall_score(labels, pred, zero_division=0), 4),
            "flag_rate_aligned": round(float(pred[labels == 0].mean()), 4),
            "flag_rate_mismatched": round(float(pred[labels == 1].mean()), 4)}


def embed(df: pd.DataFrame, cfg: dict):
    import torch
    from PIL import Image
    from transformers import CLIPModel, CLIPProcessor

    name = cfg["cscore"]["clip_model"]
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model = CLIPModel.from_pretrained(name).to(dev).eval()
    proc = CLIPProcessor.from_pretrained(name)
    T, I = [], []
    bs = cfg["cscore"]["batch_size"]
    for s in range(0, len(df), bs):
        b = df.iloc[s:s + bs]
        imgs = [Image.open(p(x)).convert("RGB") for x in b["image_path"]]
        inp = proc(text=b["title"].tolist(), images=imgs, return_tensors="pt",
                   padding=True, truncation=True, max_length=77).to(dev)
        with torch.no_grad():
            out = model(**inp)   # forward pass gives projected text_embeds/image_embeds in every version
        T.append(torch.nn.functional.normalize(out.text_embeds, dim=-1).cpu().numpy())
        I.append(torch.nn.functional.normalize(out.image_embeds, dim=-1).cpu().numpy())
        print(f"[cscore] embedded {min(s + bs, len(df))}/{len(df)}", flush=True)
    return np.concatenate(T), np.concatenate(I)


def main() -> None:
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import train_test_split

    cfg = load_config()
    cc = cfg["cscore"]
    set_seed(cfg["seed"])
    rng = np.random.default_rng(cfg["seed"])
    out_dir = cc["out_dir"]

    df = pd.read_csv(p(f"{cfg['paths']['processed']}/fnn_pairs.csv"))
    df = df[df["image_path"].map(lambda x: p(x).exists())].reset_index(drop=True)
    print(f"[cscore] {len(df)} usable pairs | {df.groupby(['site', 'fnn_label']).size().to_dict()}")

    E_t, E_i = embed(df, cfg)
    df["cscore"] = 1.0 - (E_t * E_i).sum(1)

    # mismatched: headline i with image j from the same site (j != i)
    j = derange_within(df["site"], rng)
    keep = j != np.arange(len(df))
    mis = df[keep].copy()
    mis["cscore"] = 1.0 - (E_t[keep] * E_i[j[keep]]).sum(1)
    mis["image_path"] = df["image_path"].to_numpy()[j[keep]]

    df["pair_type"], mis["pair_type"] = "aligned", "mismatched"
    df["is_mismatch"], mis["is_mismatch"] = 0, 1

    # split by HEADLINE so a headline's aligned and mismatched versions stay in the same split
    strat = df["site"] + "_" + df["fnn_label"]
    val_ids, test_ids = train_test_split(df["id"], test_size=0.5, stratify=strat,
                                         random_state=cfg["seed"])
    allp = pd.concat([df, mis], ignore_index=True)
    allp["split"] = np.where(allp["id"].isin(set(val_ids)), "valid", "test")
    allp.to_csv(p(f"{out_dir}/cscore_pairs.csv"), index=False)

    v, t = allp[allp.split == "valid"], allp[allp.split == "test"]
    t_youden = youden_threshold(v["cscore"].to_numpy(), v["is_mismatch"].to_numpy())
    t_pct = float(np.percentile(v.loc[v.is_mismatch == 0, "cscore"], cc["percentile"]))

    def stats(x):
        return {"mean": round(float(x.mean()), 4), "std": round(float(x.std()), 4),
                "min": round(float(x.min()), 4), "max": round(float(x.max()), 4)}

    al = df  # aligned only, for the fake-vs-real check
    res = {
        "clip_model": cc["clip_model"],
        "n_pairs": {"aligned": len(df), "mismatched": len(mis),
                    "valid_headlines": len(val_ids), "test_headlines": len(test_ids)},
        "cscore_distribution": {"aligned": stats(df["cscore"]), "mismatched": stats(mis["cscore"])},
        "test_auc_aligned_vs_mismatched": round(roc_auc_score(t["is_mismatch"], t["cscore"]), 4),
        "thresholds": {
            "youden_on_valid": eval_at(t["cscore"].to_numpy(), t["is_mismatch"].to_numpy(), t_youden),
            f"p{cc['percentile']}_aligned_valid": eval_at(t["cscore"].to_numpy(),
                                                         t["is_mismatch"].to_numpy(), t_pct),
            "proposal_0.62": eval_at(t["cscore"].to_numpy(), t["is_mismatch"].to_numpy(), 0.62),
        },
        "chosen_threshold": round(t_youden, 4),
        "fake_vs_real_aligned": {
            "mean_cscore_fake": round(float(al.loc[al.fnn_label == "fake", "cscore"].mean()), 4),
            "mean_cscore_real": round(float(al.loc[al.fnn_label == "real", "cscore"].mean()), 4),
            "auc_cscore_predicts_fake": round(roc_auc_score((al.fnn_label == "fake").astype(int),
                                                            al["cscore"]), 4),
        },
        "by_site_test_auc": {s: round(roc_auc_score(g["is_mismatch"], g["cscore"]), 4)
                             for s, g in t.groupby("site") if g["is_mismatch"].nunique() == 2},
    }
    write_json(res, f"{out_dir}/cscore_results.json")
    print(pd.Series(res["thresholds"]).to_string())
    print(f"[cscore] test AUC aligned vs mismatched: {res['test_auc_aligned_vs_mismatched']}")
    print(f"[cscore] chosen threshold (Youden, valid): {res['chosen_threshold']}")

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(7, 4))
        bins = np.linspace(allp["cscore"].min(), allp["cscore"].max(), 40)
        ax.hist(df["cscore"], bins=bins, alpha=0.6, label="Aligned (real pair)")
        ax.hist(mis["cscore"], bins=bins, alpha=0.6, label="Mismatched (shuffled image)")
        ax.axvline(t_youden, color="k", ls="--", label=f"Threshold {t_youden:.3f}")
        ax.set_xlabel("C-score = 1 - cos(text, image)")
        ax.set_ylabel("Pairs")
        ax.set_title("CLIP ViT-B/32 contradiction score, FakeNewsNet")
        ax.legend()
        fig.tight_layout()
        fig.savefig(p(f"{out_dir}/cscore_hist.png"), dpi=150)
    except Exception as e:  # plotting is a nice-to-have
        print("[cscore] plot skipped:", e)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/10_build_augmented.py
"""Build the augmented training corpus: LIAR train + accepted paraphrases + successful TextFooler examples.

Every augmented example keeps the gold label of the statement it came from.
Only TextFooler v2 output (names/numbers protected) is used.

Output: data/processed/corpus_aug_train.csv   (columns: id,text,label,source,label_id,orig_id)
        data/processed/corpus_aug_stats.json
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import load_config, p, write_json  # noqa: E402


def jl(path):
    f = p(path)
    return pd.DataFrame([json.loads(line) for line in open(f)]) if f.exists() else pd.DataFrame()


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--extra-tf", default=None, help="add a second round of TextFooler examples (jsonl)")
    ap.add_argument("--out", default=None, help="output csv (default corpus_aug_train.csv)")
    args = ap.parse_args()
    cfg = load_config()
    d = cfg["paths"]["processed"]
    orig = pd.read_csv(p(f"{d}/corpus_train.csv"))
    orig["source"], orig["orig_id"] = "liar", orig["id"].astype(str)

    para = jl(cfg["paraphrase"]["out_train"])
    para = para[para["status"] == "ok"] if len(para) else para
    tf = jl(cfg["textfooler"]["out_train"])
    if len(tf):
        tf = tf[(tf["status"] == "success") & (tf.get("tf_version", 0) == 2)]

    tf2 = jl(args.extra_tf) if args.extra_tf else pd.DataFrame()
    if len(tf2):
        tf2 = tf2[(tf2["status"] == "success") & (tf2.get("tf_version", 0) == 2)]
    parts = [orig[["id", "text", "label", "label_id", "source", "orig_id"]]]
    for df, src in [(para, "paraphrase"), (tf, "textfooler"), (tf2, "textfooler_r2")]:
        if src == "textfooler_r2" and not args.extra_tf:
            continue
        if len(df):
            x = pd.DataFrame({"id": {"paraphrase": "para", "textfooler": "tf", "textfooler_r2": "tf2"}[src]
                              + "-" + df["id"].astype(str), "text": df["text"],
                              "label": df["label"], "label_id": df["label_id"].astype(int),
                              "source": src, "orig_id": df["id"].astype(str)})
            parts.append(x)
        else:
            print(f"[augment] WARNING: no {src} data found")
    aug = pd.concat(parts, ignore_index=True)
    aug = aug[aug["text"].astype(str).str.strip().str.len() > 0]
    aug = aug.drop_duplicates(subset=["text"]).sample(frac=1.0, random_state=cfg["seed"]).reset_index(drop=True)
    out_csv = args.out or f"{d}/corpus_aug_train.csv"
    aug.to_csv(p(out_csv), index=False)

    n = len(aug)
    stats = {"n": n,
             "by_source": aug["source"].value_counts().to_dict(),
             "share_pct": {k: round(v / n * 100, 1) for k, v in aug["source"].value_counts().items()},
             "by_source_and_label": aug.groupby(["source", "label"]).size().unstack(fill_value=0).to_dict()}
    write_json(stats, out_csv.replace(".csv", "_stats.json") if args.out else f"{d}/corpus_aug_stats.json")
    print(f"[augment] {n} training rows | {stats['by_source']} | shares % {stats['share_pct']}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/11_train_xfakenet.py
"""Train the XFakeNet classifier (RoBERTa + token/segment attention), class-weighted CE.

--data orig : LIAR training split only            -> runs/xfakenet_noaug  (ablation: no augmentation)
--data aug  : 70% LIAR + 15% paraphrase + 15% TF  -> runs/xfakenet_v2     (robust model v2)

The best epoch is chosen by validation macro-F1. Validation and test are always the
clean LIAR splits.

Output: runs/<run>/xfakenet.pt, xfakenet_meta.json, tokenizer, metrics.json,
        test_predictions.csv, confusion_test.csv

Usage : python scripts/11_train_xfakenet.py --data aug --run-name xfakenet_v2
        python scripts/11_train_xfakenet.py --smoke        (1-minute check on a few rows)
"""
from __future__ import annotations

import argparse
import math
import shutil
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet import model as xm  # noqa: E402
from xfakenet.common import class_names, load_config, p, set_seed, write_json  # noqa: E402


def report(df: pd.DataFrame, pred: np.ndarray, classes: list[str]) -> dict:
    y = df["label_id"].to_numpy()
    return {
        "n": len(df),
        "accuracy": round(accuracy_score(y, pred), 4),
        "macro_f1": round(f1_score(y, pred, average="macro", zero_division=0), 4),
        "per_class": classification_report(y, pred, labels=list(range(len(classes))),
                                           target_names=classes, output_dict=True, zero_division=0),
    }


@torch.no_grad()
def predict(model, tok, texts, cfg, device, bs=128):
    model.eval()
    out = []
    for s in range(0, len(texts), bs):
        enc, _, _ = xm.encode(tok, texts[s:s + bs], cfg["max_len"], cfg["max_segments"], device)
        with torch.autocast(device_type="cuda", enabled=device == "cuda"):
            out.append(model(**enc)["logits"].float().cpu())
    return torch.cat(out).numpy()


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="aug", choices=["orig", "aug"])
    ap.add_argument("--run-name", default=None)
    ap.add_argument("--epochs", type=int, default=None)
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--seed", type=int, default=None)
    ap.add_argument("--out-dir", default=None, help="absolute output folder (e.g. Colab local disk instead of Drive)")
    ap.add_argument("--train-file", default=None, help="override training csv (e.g. round-2 corpus)")
    args = ap.parse_args()

    cfg = load_config()
    if args.seed is not None:
        cfg["seed"] = args.seed
    xc = cfg["xfakenet"]
    set_seed(cfg["seed"])
    classes = class_names()
    d = cfg["paths"]["processed"]
    run = args.run_name or ("xfakenet_v2" if args.data == "aug" else "xfakenet_noaug")
    if args.smoke:
        run = "xfakenet_smoke"
    out_dir = Path(args.out_dir) if args.out_dir else p(f"{cfg['paths']['runs']}/{run}")
    out_dir.mkdir(parents=True, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    train_file = args.train_file or (f"{d}/corpus_aug_train.csv" if args.data == "aug" else f"{d}/corpus_train.csv")
    train = pd.read_csv(p(train_file))
    valid = pd.read_csv(p(f"{d}/corpus_valid.csv"))
    test = pd.read_csv(p(f"{d}/corpus_test.csv"))
    epochs = args.epochs or xc["epochs"]
    if args.smoke:
        train, valid, test, epochs = train.sample(96, random_state=0), valid.head(64), test.head(64), 1
    if "source" not in train:
        train["source"] = "liar"
    print(f"[{run}] train={len(train)} {train['source'].value_counts().to_dict()} | device={device}", flush=True)

    counts = np.bincount(train["label_id"], minlength=len(classes)).astype(float)
    w = counts.sum() / (len(classes) * np.maximum(counts, 1))
    weights = torch.tensor(w / w.mean(), dtype=torch.float, device=device)

    from transformers import AutoTokenizer, get_linear_schedule_with_warmup
    tok = AutoTokenizer.from_pretrained(xc["encoder"])
    model = xm.build(xc["encoder"], len(classes), xc["max_segments"], xc["att_dim"]).to(device)

    head = [p_ for n, p_ in model.named_parameters() if not n.startswith("encoder.")]
    enc_params = [p_ for n, p_ in model.named_parameters() if n.startswith("encoder.")]
    opt = torch.optim.AdamW([{"params": enc_params, "lr": xc["lr"]},
                             {"params": head, "lr": xc["head_lr"]}], weight_decay=xc["weight_decay"])
    steps = math.ceil(len(train) / xc["batch_size"]) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(steps * xc["warmup_ratio"]), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")

    texts, labels = train["text"].tolist(), train["label_id"].to_numpy()
    import tempfile
    best_f1, best_epoch = -1.0, -1
    tmp = Path(tempfile.gettempdir()) / f"{run}_best"   # local disk, not Drive (faster)
    rng = np.random.default_rng(cfg["seed"])
    for ep in range(1, epochs + 1):
        model.train()
        order, t0, run_loss = rng.permutation(len(train)), time.time(), 0.0
        for step, s in enumerate(range(0, len(order), xc["batch_size"]), 1):
            idx = order[s:s + xc["batch_size"]]
            enc, _, _ = xm.encode(tok, [texts[i] for i in idx], xc["max_len"], xc["max_segments"], device)
            y = torch.tensor(labels[idx], device=device)
            with torch.autocast(device_type="cuda", enabled=device == "cuda"):
                logits = model(**enc)["logits"]
            loss = torch.nn.functional.cross_entropy(logits.float(), y, weight=weights)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            run_loss += loss.item()
            if step % 100 == 0:
                print(f"[{run}] epoch {ep} step {step} loss {run_loss / step:.4f}", flush=True)
        vp = predict(model, tok, valid["text"].tolist(), xc, device).argmax(-1)
        vf1 = f1_score(valid["label_id"], vp, average="macro")
        print(f"[{run}] epoch {ep} done in {time.time() - t0:.0f}s | train loss {run_loss / step:.4f} "
              f"| valid macro-F1 {vf1:.4f}", flush=True)
        if vf1 > best_f1:
            best_f1, best_epoch = vf1, ep
            xm.save(model, tok, tmp, {})

    # reload the best epoch and evaluate
    model.load_state_dict(torch.load(tmp / "xfakenet.pt", map_location=device))
    meta = {"classes": classes, "encoder": xc["encoder"], "max_len": xc["max_len"],
            "max_segments": xc["max_segments"], "att_dim": xc["att_dim"],
            "highlight_threshold": xc["highlight_threshold"], "data": args.data, "best_epoch": best_epoch}
    xm.save(model, tok, out_dir, meta)
    shutil.rmtree(tmp, ignore_errors=True)

    results = {"run_name": run, "data": args.data, "seed": cfg["seed"], "train_file": train_file, "train_sources": train["source"].value_counts().to_dict(),
               "config": xc, "epochs": epochs, "best_epoch": best_epoch, "lr": xc["lr"]}
    for name, df in [("valid", valid), ("test", test)]:
        pred = predict(model, tok, df["text"].tolist(), xc, device).argmax(-1)
        results[name] = report(df.reset_index(drop=True), pred, classes)
        print(f"[{run}] {name}: acc={results[name]['accuracy']} macro_f1={results[name]['macro_f1']}", flush=True)
        if name == "test":
            cm = confusion_matrix(df["label_id"], pred, labels=list(range(len(classes))))
            pd.DataFrame(cm, index=classes, columns=classes).to_csv(out_dir / "confusion_test.csv")
            pd.DataFrame({"id": df["id"], "text": df["text"], "label": df["label"],
                          "pred": [classes[i] for i in pred]}).to_csv(out_dir / "test_predictions.csv", index=False)
    write_json(results, out_dir / "metrics.json")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/12_evaluate_robustness.py
"""Compare models on clean, paraphrased and adversarial LIAR test text.

Models: baseline (week 1), XFakeNet without augmentation, XFakeNet v2 (augmented).

1. Clean test        : accuracy / macro-F1 on the 1,283 LIAR test statements.
2. Paraphrased test  : accuracy on the Gemini-paraphrased test statements vs their originals,
                       and how often the prediction stays the same.
3. TextFooler (fresh): each model attacked by its OWN TextFooler run on the same 499 test
                       statements. This is the fair robustness number.
4. TextFooler (transfer, reference only): accuracy on the adversarial texts that fooled the
                       baseline. Easier for other models, so it overstates robustness.

Output: results/week3/robustness.json, results/week3/robustness.md
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import load_config, p, write_json  # noqa: E402

MODELS = {  # name: (kind, model dir, fresh TextFooler test file)
    "Baseline (RoBERTa)": ("hf", "runs/baseline/best", "data/augment/textfooler_test.jsonl"),
    "XFakeNet, no augmentation": ("xfakenet", "runs/xfakenet_noaug",
                                  "data/augment/textfooler_test_xfakenet_noaug.jsonl"),
    "XFakeNet v2 (augmented)": ("xfakenet", "runs/xfakenet_v2", "data/augment/textfooler_test_xfakenet_v2.jsonl"),
}
METRICS = {"Baseline (RoBERTa)": "runs/baseline/metrics.json",
           "XFakeNet, no augmentation": "runs/xfakenet_noaug/metrics.json",
           "XFakeNet v2 (augmented)": "runs/xfakenet_v2/metrics.json"}


def jl(path):
    f = p(path)
    return pd.DataFrame([json.loads(line) for line in open(f)]) if f.exists() else pd.DataFrame()


def predictor(kind: str, model_dir: str, cfg: dict):
    if kind == "xfakenet":
        from xfakenet.model import Predictor
        return Predictor(p(model_dir)).predict_proba
    from xfakenet.textfooler import make_predict
    return make_predict(str(p(model_dir)), cfg["baseline"]["max_len"])


def main() -> None:
    cfg = load_config()
    out_dir = cfg["week3"]["out_dir"]
    para = jl(cfg["paraphrase"]["out_test"])
    para = para[para["status"] == "ok"].reset_index(drop=True) if len(para) else para
    base_adv = jl(cfg["textfooler"]["out_test"])
    base_adv = base_adv[(base_adv["status"] == "success") & (base_adv.get("tf_version", 0) == 2)] \
        if len(base_adv) else base_adv

    res = {}
    for name, (kind, mdir, tf_file) in MODELS.items():
        if not p(mdir).exists():
            print(f"[robust] {name}: model not found at {mdir}, skipped")
            continue
        r = {}
        m = json.load(open(p(METRICS[name])))
        r["clean_accuracy"], r["clean_macro_f1"] = m["test"]["accuracy"], m["test"]["macro_f1"]
        pred = predictor(kind, mdir, cfg)

        if len(para):
            po = pred(para["original"].tolist()).argmax(1)
            pp = pred(para["text"].tolist()).argmax(1)
            y = para["label_id"].to_numpy()
            r["paraphrase"] = {"n": len(para), "acc_original": round(accuracy_score(y, po), 4),
                               "acc_paraphrased": round(accuracy_score(y, pp), 4),
                               "prediction_unchanged": round(float((po == pp).mean()), 4)}

        t = jl(tf_file)
        if len(t):
            vc = t["status"].value_counts()
            s, f, n = int(vc.get("success", 0)), int(vc.get("failed", 0)), len(t)
            r["textfooler_fresh"] = {"n": n, "clean_acc_on_sample": round((s + f) / n, 4),
                                     "acc_under_attack": round(f / n, 4),
                                     "attack_success_rate": round(s / max(1, s + f), 4),
                                     "avg_words_changed": round(float(t.loc[t.status == "success",
                                                                          "words_changed"].mean()), 2)
                                     if s else None}
        if len(base_adv):
            pa = pred(base_adv["text"].tolist()).argmax(1)
            r["textfooler_transfer_from_baseline"] = {
                "n": len(base_adv), "acc": round(accuracy_score(base_adv["label_id"], pa), 4)}
        res[name] = r
        print(f"[robust] {name}: {r}", flush=True)

    write_json(res, f"{out_dir}/robustness.json")

    L = ["| Model | Clean acc | Clean macro-F1 | Paraphrased acc (orig → para) | Prediction unchanged "
         "| TextFooler: acc under attack | Attack success rate | Transfer acc* |",
         "|---|---|---|---|---|---|---|---|"]
    for name, r in res.items():
        pa = r.get("paraphrase", {})
        tf_ = r.get("textfooler_fresh", {})
        tr = r.get("textfooler_transfer_from_baseline", {})
        L.append(f"| {name} | {r['clean_accuracy']:.3f} | {r['clean_macro_f1']:.3f} | "
                 f"{pa.get('acc_original', float('nan')):.3f} → {pa.get('acc_paraphrased', float('nan')):.3f} | "
                 f"{pa.get('prediction_unchanged', float('nan')) * 100:.0f}% | "
                 f"{tf_.get('acc_under_attack', float('nan')):.3f} | "
                 f"{tf_.get('attack_success_rate', float('nan')) * 100:.0f}% | {tr.get('acc', float('nan')):.3f} |")
    L.append("")
    L.append("*Transfer = accuracy on the adversarial texts that fooled the baseline (reference only; "
             "the fair robustness number is the model's own TextFooler attack).")
    open(p(f"{out_dir}/robustness.md"), "w").write("\n".join(L))
    print("\n".join(L))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/14_modality_flag.py
"""Module 1 (modality level): flag each FakeNewsNet item as TEXT, IMAGE or CROSS-MODAL.

Outline rule: CROSS-MODAL when the C-score exceeds its calibrated threshold; otherwise
TEXT or IMAGE, depending on which modality carries more of the evidence.

Evidence per modality comes from two small fake-vs-real classifiers on FakeNewsNet:
  text  : logistic regression on the CLIP embedding of the headline
  image : logistic regression on the CLIP embedding of the image
A fusion model combines their (out-of-fold) logits; each modality's contribution is
coefficient x its logit. The bigger absolute contribution decides TEXT vs IMAGE.

All probabilities are out-of-fold (5-fold CV), so no item is scored by a model that saw it.
This also gives the C-score ablation on FakeNewsNet: fake/real AUC with and without it.

Output: results/week3/modality_flags.csv, results/week3/modality_results.json
"""
from __future__ import annotations

import importlib.util
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import load_config, p, set_seed, write_json  # noqa: E402

ROOT = Path(__file__).resolve().parent


def clip_embed(df: pd.DataFrame, cfg: dict):
    spec = importlib.util.spec_from_file_location("cscore", ROOT / "08_clip_cscore.py")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.embed(df, cfg)


def oof_logit(X: np.ndarray, y: np.ndarray, folds, C: float = 1.0) -> np.ndarray:
    out = np.zeros(len(y))
    for tr, te in folds:
        m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
        m.fit(X[tr], y[tr])
        out[te] = m.decision_function(X[te])
    return out


def main() -> None:
    cfg = load_config()
    set_seed(cfg["seed"])
    out_dir = cfg["week3"]["out_dir"]
    mc = cfg["modality"]

    df = pd.read_csv(p(f"{cfg['paths']['processed']}/fnn_pairs.csv"))
    df = df[df["image_path"].map(lambda x: p(x).exists())].reset_index(drop=True)
    thr = json.load(open(p(f"{cfg['cscore']['out_dir']}/cscore_results.json")))["chosen_threshold"]
    T, I = clip_embed(df, cfg)
    df["cscore"] = 1.0 - (T * I).sum(1)
    y = (df["fnn_label"] == "fake").astype(int).to_numpy()

    strat = df["site"] + "_" + df["fnn_label"]
    folds = list(StratifiedKFold(5, shuffle=True, random_state=cfg["seed"]).split(df, strat))
    zt = oof_logit(T, y, folds, mc["C"])
    zi = oof_logit(I, y, folds, mc["C"])
    zc = df["cscore"].to_numpy()

    def fused(cols):
        X = np.column_stack(cols)
        return oof_logit(X, y, folds, 1.0)

    auc = lambda z: round(float(roc_auc_score(y, z)), 4)  # noqa: E731
    res_auc = {"text_only": auc(zt), "image_only": auc(zi), "cscore_only": auc(zc),
               "text+image": auc(fused([zt, zi])), "text+image+cscore": auc(fused([zt, zi, zc]))}

    # contributions for the flag: fusion model fit on all out-of-fold logits
    fz = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(np.column_stack([zt, zi]), y)
    sc, lr = fz.named_steps["standardscaler"], fz.named_steps["logisticregression"]
    Z = sc.transform(np.column_stack([zt, zi]))
    contrib_text, contrib_img = lr.coef_[0][0] * Z[:, 0], lr.coef_[0][1] * Z[:, 1]
    df["p_fake"] = fz.predict_proba(np.column_stack([zt, zi]))[:, 1]
    df["contrib_text"], df["contrib_image"] = contrib_text, contrib_img
    df["modality_flag"] = np.where(df["cscore"] > thr, "CROSS-MODAL",
                                   np.where(np.abs(contrib_text) >= np.abs(contrib_img), "TEXT", "IMAGE"))
    df[["id", "title", "fnn_label", "site", "image_path", "cscore", "p_fake", "contrib_text",
        "contrib_image", "modality_flag"]].to_csv(p(f"{out_dir}/modality_flags.csv"), index=False)

    res = {"n": len(df), "cscore_threshold": thr,
           "fake_vs_real_auc_5fold": res_auc,
           "flag_counts": df["modality_flag"].value_counts().to_dict(),
           "flag_by_label": df.groupby(["fnn_label", "modality_flag"]).size().unstack(fill_value=0).to_dict(),
           "fusion_weights": {"text": round(float(lr.coef_[0][0]), 4), "image": round(float(lr.coef_[0][1]), 4)}}
    write_json(res, f"{out_dir}/modality_results.json")
    print(json.dumps(res, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/15_faithfulness.py
"""Week 4: are the explanations faithful? (deletion test + exact Shapley/SHAP agreement)

1. Word deletion (comprehensiveness): remove the top-k highlighted words and measure how much
   the probability of the predicted class drops. Compare with removing k RANDOM words
   (average of 3 draws). Faithful highlights should cause a clearly larger drop.
   Conditions: k = 1, k = 3, and "all highlighted words" (importance >= 0.65).
2. Segment deletion (statements with 2+ segments): remove the most-attended segment vs a
   random other segment.
3. SHAP agreement (statements with 2+ segments): exact Shapley values of each segment
   (all 2^K subsets, K <= 8, segment removed = deleted from the text). Report how often the
   Shapley top segment equals the attention top segment, vs the 1/K chance level, and the
   mean Spearman rank correlation for statements with 3+ segments.

Output: results/week4/faithfulness.json, results/week4/faithfulness.md
"""
from __future__ import annotations

import itertools
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import load_config, p, write_json  # noqa: E402

MODELS = {"XFakeNet v2 (augmented)": "runs/xfakenet_v2", "XFakeNet, no augmentation": "runs/xfakenet_noaug"}


def drop_words(words: list[str], remove: set[int]) -> str:
    return " ".join(w for i, w in enumerate(words) if i not in remove)


def word_deletion(pred, ex: list[dict], thr: float, rng: np.random.Generator, n_random: int = 3) -> dict:
    """Mean drop in predicted-class probability: top-k words vs random-k words."""
    conds = {"k=1": 1, "k=3": 3, "highlighted": None}
    texts, meta = [], []  # meta: (example idx, condition, kind)
    for i, e in enumerate(ex):
        words = [w["w"] for w in e["words"]]
        weights = np.array([w["weight"] for w in e["words"]])
        if len(words) < 4:
            continue
        order = list(np.argsort(-weights))
        for name, k in conds.items():
            top = set(order[:k]) if k else {j for j in range(len(words)) if weights[j] >= thr}
            if not top or len(top) >= len(words):
                continue
            texts.append(drop_words(words, top))
            meta.append((i, name, "top"))
            for _ in range(n_random):
                rnd = set(rng.choice(len(words), len(top), replace=False).tolist())
                texts.append(drop_words(words, rnd))
                meta.append((i, name, "random"))
    probs = pred.predict_proba(texts) if texts else np.zeros((0, 1))
    classes = list(ex[0]["probs"]) if ex else []
    rows = []
    for (i, name, kind), pr in zip(meta, probs):
        c = classes.index(ex[i]["pred"])
        p0 = ex[i]["probs"][ex[i]["pred"]]
        rows.append({"i": i, "cond": name, "kind": kind, "drop": p0 - pr[c], "flip": int(pr.argmax() != c)})
    df = pd.DataFrame(rows)
    out = {}
    for name in conds:
        d = df[df.cond == name]
        if not len(d):
            continue
        top, rnd = d[d.kind == "top"], d[d.kind == "random"]
        out[name] = {"n": int(top.i.nunique()),
                     "drop_top": round(float(top["drop"].mean()), 4),
                     "drop_random": round(float(rnd["drop"].mean()), 4),
                     "flip_rate_top": round(float(top["flip"].mean()), 4),
                     "flip_rate_random": round(float(rnd["flip"].mean()), 4),
                     # paired: per statement, does removing the top words hurt more than random?
                     "top_beats_random_pct": round(float(
                         (top.set_index("i")["drop"] > rnd.groupby("i")["drop"].mean()).mean() * 100), 1)}
    return out


def shapley(values: dict, k: int) -> np.ndarray:
    """Exact Shapley values from v(S) for all subsets S (tuples of segment indices)."""
    phi = np.zeros(k)
    for i in range(k):
        others = [j for j in range(k) if j != i]
        for r in range(len(others) + 1):
            w = math.factorial(r) * math.factorial(k - r - 1) / math.factorial(k)
            for S in itertools.combinations(others, r):
                phi[i] += w * (values[tuple(sorted(S + (i,)))] - values[S])
    return phi


def segment_tests(pred, ex: list[dict], rng: np.random.Generator) -> dict:
    multi = [(i, e) for i, e in enumerate(ex) if 2 <= len(e["segments"]) <= 8]
    classes = list(ex[0]["probs"]) if ex else []
    texts, keys = [], []
    for i, e in multi:
        segs = [s["text"] for s in e["segments"]]
        k = len(segs)
        for r in range(k + 1):
            for S in itertools.combinations(range(k), r):
                texts.append(" ".join(segs[j] for j in S))
                keys.append((i, S))
    probs = pred.predict_proba(texts) if texts else np.zeros((0, 1))
    v = {}
    for (i, S), pr in zip(keys, probs):
        v.setdefault(i, {})[S] = pr[classes.index(ex[i]["pred"])]

    agree, chance, rhos, d_top, d_rnd = [], [], [], [], []
    for i, e in multi:
        k = len(e["segments"])
        att = np.array([s["weight"] for s in e["segments"]])
        phi = shapley(v[i], k)
        agree.append(int(np.argmax(phi) == np.argmax(att)))
        chance.append(1 / k)
        if k >= 3:
            r = spearmanr(att, phi).correlation
            if not np.isnan(r):
                rhos.append(r)
        full = tuple(range(k))
        top = int(np.argmax(att))
        other = int(rng.choice([j for j in range(k) if j != top]))
        d_top.append(v[i][full] - v[i][tuple(j for j in full if j != top)])
        d_rnd.append(v[i][full] - v[i][tuple(j for j in full if j != other)])
    n = len(multi)
    return {
        "n_multi_segment": n,
        "segment_deletion": {"drop_top_segment": round(float(np.mean(d_top)), 4) if n else None,
                             "drop_random_segment": round(float(np.mean(d_rnd)), 4) if n else None},
        "shap_agreement": {"top1_agreement_pct": round(float(np.mean(agree) * 100), 1) if n else None,
                           "chance_pct": round(float(np.mean(chance) * 100), 1) if n else None,
                           "spearman_mean_3plus": round(float(np.mean(rhos)), 3) if rhos else None,
                           "n_3plus": len(rhos)},
    }


def main() -> None:
    from xfakenet.model import Predictor
    cfg = load_config()
    out_dir = cfg["week4"]["out_dir"]
    thr = cfg["xfakenet"]["highlight_threshold"]
    test = pd.read_csv(p(f"{cfg['paths']['processed']}/corpus_test.csv"))
    res = {}
    for name, mdir in MODELS.items():
        if not p(mdir).exists():
            continue
        rng = np.random.default_rng(cfg["seed"])
        pred = Predictor(p(mdir))
        ex = pred.explain(test["text"].tolist(), thr)
        res[name] = {"word_deletion": word_deletion(pred, ex, thr, rng), **segment_tests(pred, ex, rng)}
        print(f"[faith] {name}: {res[name]}", flush=True)
    write_json(res, f"{out_dir}/faithfulness.json")

    L = ["| Model | Removed | Drop (top) | Drop (random) | Flip rate top / random | Top hurts more than random |",
         "|---|---|---|---|---|---|"]
    for name, r in res.items():
        for cond, w in r["word_deletion"].items():
            L.append(f"| {name} | {cond} words | {w['drop_top']:.3f} | {w['drop_random']:.3f} | "
                     f"{w['flip_rate_top']*100:.0f}% / {w['flip_rate_random']*100:.0f}% | {w['top_beats_random_pct']}% |")
        sd = r["segment_deletion"]
        if sd["drop_top_segment"] is not None:
            L.append(f"| {name} | top segment | {sd['drop_top_segment']:.3f} | {sd['drop_random_segment']:.3f} | – | – |")
    L += ["", "| Model | Statements (2+ segments) | Attention top segment = SHAP top segment | Chance | "
              "Spearman (3+ segments) |", "|---|---|---|---|---|"]
    for name, r in res.items():
        a = r["shap_agreement"]
        L.append(f"| {name} | {r['n_multi_segment']} | {a['top1_agreement_pct']}% | {a['chance_pct']}% | "
                 f"{a['spearman_mean_3plus']} (n={a['n_3plus']}) |")
    open(p(f"{out_dir}/faithfulness.md"), "w").write("\n".join(L))
    print("\n".join(L))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/16_deception.py
"""Week 4: six-category deception-pattern attribution for image-text pairs.

Stages (run in order; each is skipped if its inputs are missing):
  label : Gemini (multimodal) labels each pair with one of six deception patterns or
          "consistent". Pairs = all 389 real FakeNewsNet pairs + 200 shuffled (mismatched) pairs.
  train : a small classifier (logistic regression on CLIP text/image features + C-score)
          learns the six patterns from Gemini's labels; 5-fold cross-validated.
          This classifier is what the pipeline uses for CROSS-MODAL items (no API at run time).
  page  : builds results/week4/handcheck.html with 100 pairs for you to verify by hand
          (works offline in any browser; download your answers as CSV at the end).
  score : after you upload the CSV, measures Gemini's and the classifier's agreement with you.

Usage: python scripts/16_deception.py --stage all
       python scripts/16_deception.py --stage score
"""
from __future__ import annotations

import argparse
import base64
import html
import io
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import load_config, p, set_seed, write_json  # noqa: E402

CATS = {
    "out_of_context": "Genuine photo, but from a different event, time or story than the headline",
    "misattributed_entity": "Shows a different person, group or organisation than the headline names",
    "wrong_location_or_event": "Shows a different place or kind of event than the headline describes",
    "exaggeration": "Image and headline differ sharply in drama or emotion (one sensationalises the other)",
    "manipulated_or_synthetic": "Looks edited, composited, a meme, a screenshot with added text, or AI-generated",
    "irrelevant_stock": "Generic stock or decorative image with no real connection to the specific claim",
    "consistent": "Image plausibly matches the headline (no deception pattern)",
}
PATTERNS = [c for c in CATS if c != "consistent"]

PROMPT = """You are annotating news headline + image pairs for research on image-text deception.
For each item, choose exactly ONE category:
{cats}

Judge only the relationship between the headline and the image (not whether the claim is true).
Return a JSON array, one object per item, same order:
[{{"i": <index>, "category": "<category key>", "confidence": <0-1>, "reason": "<max 15 words>"}}, ...]
"""


def items(cfg: dict) -> pd.DataFrame:
    """Real pairs + a sample of shuffled pairs (from the C-score step)."""
    dc = cfg["deception"]
    al = pd.read_csv(p(f"{cfg['paths']['processed']}/fnn_pairs.csv"))
    al = al[al["image_path"].map(lambda x: p(x).exists())]
    al = al.assign(pair_type="aligned")[["id", "title", "image_path", "fnn_label", "site", "pair_type"]]
    parts = [al]
    cp = p(f"{cfg['cscore']['out_dir']}/cscore_pairs.csv")
    if cp.exists():
        mis = pd.read_csv(cp)
        mis = mis[mis["pair_type"] == "mismatched"]
        mis = mis.sample(min(len(mis), dc["n_mismatched"]), random_state=cfg["seed"])
        parts.append(mis[["id", "title", "image_path", "fnn_label", "site", "pair_type"]])
    df = pd.concat(parts, ignore_index=True)
    df["key"] = df["id"].astype(str) + "|" + df["pair_type"]
    return df.reset_index(drop=True)


def jpeg_bytes(path, side: int) -> bytes:
    from PIL import Image
    im = Image.open(p(path)).convert("RGB")
    im.thumbnail((side, side))
    buf = io.BytesIO()
    im.save(buf, "JPEG", quality=85)
    return buf.getvalue()


# ---------------- stage: label ----------------
def stage_label(cfg: dict, df: pd.DataFrame) -> None:
    dc = cfg["deception"]
    key = os.environ.get("GEMINI_API_KEY", "")
    if not key:
        print("[deception] no GEMINI_API_KEY in the environment; labelling skipped")
        return
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=key)
    out = p(dc["labels"])
    done = {json.loads(l)["key"] for l in open(out)} if out.exists() else set()
    todo = df[~df["key"].isin(done)]
    print(f"[deception] {len(df)} pairs, {len(done)} labelled, {len(todo)} to go", flush=True)
    cats = "\n".join(f'- "{k}": {v}' for k, v in CATS.items())
    model = None
    gcfg = types.GenerateContentConfig(response_mime_type="application/json", temperature=0.2)
    bs = dc["batch_size"]
    with open(out, "a") as f:
        for s in range(0, len(todo), bs):
            b = todo.iloc[s:s + bs]
            contents = [PROMPT.format(cats=cats)]
            for i, r in enumerate(b.itertuples(index=False)):
                contents += [f"Item {i}. Headline: {r.title}",
                             types.Part.from_bytes(data=jpeg_bytes(r.image_path, dc["image_side"]),
                                                   mime_type="image/jpeg")]
            got = None
            for m in ([model] if model else cfg["paraphrase"]["models"]):
                for attempt in range(5):
                    try:
                        resp = client.models.generate_content(model=m, contents=contents, config=gcfg)
                        got = {int(o["i"]): o for o in json.loads(resp.text)}
                        model = m
                        break
                    except Exception as e:
                        msg = str(e)
                        if "404" in msg or "NOT_FOUND" in msg or "402" in msg or "credits" in msg.lower():
                            print(f"[deception] {m}: {msg[:120]}", flush=True)
                            break
                        time.sleep(min(60, 5 * 2 ** attempt))
                if got is not None:
                    break
            if got is None:
                print(f"[deception] batch {s // bs} failed; will retry on the next run", flush=True)
                continue
            for i, r in enumerate(b.itertuples(index=False)):
                o = got.get(i, {})
                cat = str(o.get("category", "")).strip()
                f.write(json.dumps({"key": r.key, "category": cat if cat in CATS else "invalid",
                                    "confidence": o.get("confidence"), "reason": o.get("reason", ""),
                                    "model": model}) + "\n")
            f.flush()
            print(f"[deception] labelled {min(s + bs, len(todo))}/{len(todo)}", flush=True)
            time.sleep(dc["sleep_s"])


def load_labels(cfg: dict, df: pd.DataFrame) -> pd.DataFrame:
    f = p(cfg["deception"]["labels"])
    if not f.exists():
        return pd.DataFrame()
    lab = pd.DataFrame([json.loads(l) for l in open(f)]).drop_duplicates("key", keep="last")
    return df.merge(lab, on="key", how="inner")


# ---------------- stage: train ----------------
def stage_train(cfg: dict, df: pd.DataFrame) -> dict:
    import joblib
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, f1_score
    from sklearn.model_selection import StratifiedKFold, cross_val_predict
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from xfakenet.clip_utils import Clip, pair_features

    lab = load_labels(cfg, df)
    if not len(lab):
        print("[deception] no labels yet; training skipped")
        return {}
    res = {"n_labelled": len(lab),
           "label_counts": lab["category"].value_counts().to_dict(),
           "label_counts_by_pair_type": lab.groupby(["pair_type", "category"]).size().unstack(fill_value=0)
           .to_dict(orient="index")}

    feats_f = p(cfg["deception"]["features"])
    T, I = Clip(cfg["cscore"]["clip_model"]).embed(lab["title"].tolist(), [p(x) for x in lab["image_path"]])
    np.savez(feats_f, key=lab["key"].to_numpy(), T=T, I=I)
    X = pair_features(T, I)

    pat = lab["category"].isin(PATTERNS).to_numpy()
    y = lab.loc[pat, "category"].to_numpy()
    counts = pd.Series(y).value_counts()
    keep_cls = counts[counts >= cfg["deception"]["min_per_class"]].index.tolist()
    m = np.isin(y, keep_cls)
    Xp, yp = X[pat][m], y[m]
    res["classifier_classes"] = keep_cls
    res["dropped_rare_classes"] = counts[counts < cfg["deception"]["min_per_class"]].to_dict()
    if len(keep_cls) < 2:
        print("[deception] fewer than 2 pattern classes with enough examples; classifier skipped")
        write_json(res, f"{cfg['week4']['out_dir']}/deception_results.json")
        return res
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=cfg["deception"]["C"], max_iter=5000,
                                                             class_weight="balanced"))
    n_splits = int(min(5, counts[keep_cls].min()))
    cv = StratifiedKFold(n_splits, shuffle=True, random_state=cfg["seed"])
    oof = cross_val_predict(clf, Xp, yp, cv=cv)
    majority = pd.Series(yp).value_counts(normalize=True).iloc[0]
    res["classifier_cv"] = {"n": int(len(yp)), "folds": n_splits,
                            "accuracy": round(float(accuracy_score(yp, oof)), 4),
                            "macro_f1": round(float(f1_score(yp, oof, average="macro")), 4),
                            "majority_class_accuracy": round(float(majority), 4)}
    keys = lab.loc[pat, "key"].to_numpy()[m]
    pd.DataFrame({"key": keys, "gemini_category": yp, "oof_pred": oof}).to_csv(
        p(cfg["deception"]["oof"]), index=False)
    clf.fit(Xp, yp)
    joblib.dump({"model": clf, "classes": keep_cls}, p(cfg["deception"]["model"]))
    print(f"[deception] classifier CV: {res['classifier_cv']}", flush=True)
    write_json(res, f"{cfg['week4']['out_dir']}/deception_results.json")
    return res


# ---------------- stage: page ----------------
PAGE_CSS = """
:root{--bg:#f7f6f2;--card:#fff;--ink:#1c1c1a;--muted:#66655f;--line:#e2dfd6;--acc:#b45309;--ok:#2f7d4f;--bad:#b3261e}
@media (prefers-color-scheme:dark){:root{--bg:#161614;--card:#21201d;--ink:#eceae4;--muted:#a3a198;--line:#34332e;--acc:#f59e0b;--ok:#6cc58f;--bad:#f2867e}}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font:15px/1.5 -apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif}
header{position:sticky;top:0;background:var(--bg);border-bottom:1px solid var(--line);padding:12px 16px;z-index:2}
.wrap{max-width:860px;margin:0 auto}h1{font-size:20px;margin:0}
.bar{display:flex;gap:12px;align-items:center;flex-wrap:wrap;margin-top:8px}
.prog{flex:1;min-width:160px;height:8px;background:var(--line);border-radius:4px;overflow:hidden}.prog i{display:block;height:100%;background:var(--acc)}
button{font:inherit;padding:8px 14px;border-radius:8px;border:1px solid var(--acc);background:var(--acc);color:#fff;cursor:pointer}
main{padding:16px}details{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:10px 14px;margin-bottom:16px}
details li{margin:2px 0}.card{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:14px;margin-bottom:14px;
display:grid;grid-template-columns:220px 1fr;gap:14px}.card.done{border-color:var(--ok)}
.card img{width:100%;border-radius:6px;object-fit:cover;max-height:220px}
.n{font-size:12px;color:var(--muted)}.hl{font-weight:600;margin:2px 0 8px}.gem{font-size:14px;margin-bottom:8px}
.gem b{color:var(--acc)}.why{color:var(--muted);font-size:13px}label{margin-right:14px;cursor:pointer}
select{font:inherit;margin-top:6px;padding:4px;border-radius:6px;border:1px solid var(--line);background:var(--card);color:var(--ink)}
@media (max-width:600px){.card{grid-template-columns:1fr}}
"""

PAGE_JS = """
const KEY='xfn-handcheck-v1';let S={};try{S=JSON.parse(localStorage.getItem(KEY)||'{}')}catch(e){S={}}
function save(){try{localStorage.setItem(KEY,JSON.stringify(S))}catch(e){}upd()}
function upd(){const n=ITEMS.filter(it=>S[it.key]&&(S[it.key].v==='correct'||(S[it.key].v==='wrong'&&S[it.key].c))).length;
document.getElementById('cnt').textContent=n+' / '+ITEMS.length+' checked';document.getElementById('pbar').style.width=(100*n/ITEMS.length)+'%';
ITEMS.forEach(it=>{const el=document.getElementById('c-'+it.idx);const s=S[it.key];el.classList.toggle('done',!!s&&(s.v==='correct'||(s.v==='wrong'&&!!s.c)))})}
function setV(k,v){S[k]=S[k]||{};S[k].v=v;if(v==='correct')S[k].c='';document.getElementById('s-'+k).disabled=(v!=='wrong');save()}
function setC(k,c){S[k]=S[k]||{v:'wrong'};S[k].c=c;save()}
function dl(){const rows=[['key','gemini_category','verdict','human_category']];
ITEMS.forEach(it=>{const s=S[it.key]||{};rows.push([it.key,it.gem,s.v||'',s.v==='correct'?it.gem:(s.c||'')])});
const csv=rows.map(r=>r.map(x=>'"'+String(x).replace(/"/g,'""')+'"').join(',')).join('\\n');
const a=document.createElement('a');a.href=URL.createObjectURL(new Blob([csv],{type:'text/csv'}));a.download='handcheck_results.csv';a.click()}
window.addEventListener('DOMContentLoaded',()=>{ITEMS.forEach(it=>{const s=S[it.key];if(!s)return;
const r=document.querySelector('input[name="v-'+it.idx+'"][value="'+s.v+'"]');if(r)r.checked=true;
const sel=document.getElementById('s-'+it.key);sel.disabled=s.v!=='wrong';if(s.c)sel.value=s.c});upd()});
"""


def stage_page(cfg: dict, df: pd.DataFrame) -> None:
    lab = load_labels(cfg, df)
    lab = lab[lab["category"].isin(CATS)]
    if not len(lab):
        print("[deception] no labels yet; hand-check page skipped")
        return
    n = cfg["deception"]["handcheck_n"]
    rng = np.random.default_rng(cfg["seed"])
    # at least a few of every category, the rest proportional
    picks = []
    for c, g in lab.groupby("category"):
        picks += g.sample(min(len(g), 5), random_state=cfg["seed"]).index.tolist()
    rest = lab.drop(picks)
    picks += rest.sample(min(len(rest), max(0, n - len(picks))), random_state=cfg["seed"]).index.tolist()
    sample = lab.loc[picks].sample(frac=1.0, random_state=cfg["seed"]).head(n).reset_index(drop=True)
    sample[["key", "category"]].to_csv(p(cfg["deception"]["handcheck_sample"]), index=False)

    js_items, cards = [], []
    opts = "".join(f'<option value="{k}">{k.replace("_", " ")}</option>' for k in CATS)
    for i, r in sample.iterrows():
        b64 = base64.b64encode(jpeg_bytes(r.image_path, 360)).decode()
        k = html.escape(r.key, quote=True)
        js_items.append({"idx": i, "key": r.key, "gem": r.category})
        cards.append(
            f'<div class="card" id="c-{i}"><img alt="" src="data:image/jpeg;base64,{b64}">'
            f'<div><div class="n">#{i + 1}</div><div class="hl">{html.escape(str(r.title))}</div>'
            f'<div class="gem">Gemini: <b>{r.category.replace("_", " ")}</b> '
            f'<span class="why">— {html.escape(str(r.reason))}</span></div>'
            f'<label><input type="radio" name="v-{i}" value="correct" onchange="setV(\'{k}\',\'correct\')"> Correct</label>'
            f'<label><input type="radio" name="v-{i}" value="wrong" onchange="setV(\'{k}\',\'wrong\')"> Wrong, should be:</label>'
            f'<br><select id="s-{k}" disabled onchange="setC(\'{k}\',this.value)"><option value="">choose…</option>{opts}</select>'
            f'</div></div>')
    legend = "".join(f"<li><b>{k.replace('_', ' ')}</b>: {html.escape(v)}</li>" for k, v in CATS.items())
    page = (f"<!doctype html><html lang='en'><head><meta charset='utf-8'><meta name='viewport' "
            f"content='width=device-width,initial-scale=1'><title>Deception hand-check</title><style>{PAGE_CSS}</style>"
            f"</head><body><header><div class='wrap'><h1>Hand-check Gemini's deception labels</h1><div class='bar'>"
            f"<div class='prog'><i id='pbar' style='width:0'></i></div><span id='cnt'></span>"
            f"<button onclick='dl()'>Download results (CSV)</button></div></div></header><main class='wrap'>"
            f"<details open><summary><b>How to check</b> (your answers are saved in this browser as you go)</summary>"
            f"<p>For each pair, look at the headline and image and decide whether Gemini's category is right. "
            f"If not, pick the right one. When finished, click <b>Download results (CSV)</b> and upload that file "
            f"in the Week 4 notebook.</p><ul>{legend}</ul></details>{''.join(cards)}</main>"
            f"<script>const ITEMS={json.dumps(js_items)};{PAGE_JS}</script></body></html>")
    out = p(f"{cfg['week4']['out_dir']}/handcheck.html")
    out.write_text(page, encoding="utf-8")
    print(f"[deception] hand-check page: {out} ({len(sample)} pairs)")


# ---------------- stage: score ----------------
def stage_score(cfg: dict) -> None:
    from sklearn.metrics import cohen_kappa_score
    f = p(cfg["deception"]["handcheck_results"])
    if not f.exists():
        print("[deception] no hand-check results uploaded yet; scoring skipped")
        return
    h = pd.read_csv(f)
    h = h[h["verdict"].isin(["correct", "wrong"]) & h["human_category"].fillna("").astype(str).str.len().gt(0)]
    resf = p(f"{cfg['week4']['out_dir']}/deception_results.json")
    res = json.load(open(resf)) if resf.exists() else {}
    hc = {"n_checked": int(len(h)),
          "gemini_accuracy": round(float((h["verdict"] == "correct").mean()), 4) if len(h) else None,
          "gemini_kappa": round(float(cohen_kappa_score(h["gemini_category"], h["human_category"])), 4)
          if len(h) > 1 else None,
          "pattern_vs_consistent_agreement": round(float(
              ((h["gemini_category"] == "consistent") == (h["human_category"] == "consistent")).mean()), 4)
          if len(h) else None}
    oof = p(cfg["deception"]["oof"])
    if oof.exists():
        o = pd.read_csv(oof).merge(h[["key", "human_category"]], on="key")
        if len(o):
            hc["classifier_accuracy_vs_human"] = round(float((o["oof_pred"] == o["human_category"]).mean()), 4)
            hc["classifier_n"] = int(len(o))
    res["handcheck"] = hc
    write_json(res, resf)
    print(f"[deception] hand-check: {hc}")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage", default="all", choices=["all", "label", "train", "page", "score"])
    args = ap.parse_args()
    cfg = load_config()
    set_seed(cfg["seed"])
    df = items(cfg)
    if args.stage in ("all", "label"):
        stage_label(cfg, df)
    if args.stage in ("all", "train"):
        stage_train(cfg, df)
    if args.stage in ("all", "page"):
        stage_page(cfg, df)
    if args.stage in ("all", "score"):
        stage_score(cfg)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/17_pipeline_demo.py
"""Week 4: fit the deployable modality models and run the integrated pipeline on example inputs.

1. Fits text-only and image-only logistic models (CLIP features, all 389 FakeNewsNet pairs)
   and the fusion weights used for the TEXT / IMAGE flag; saves runs/modality.joblib.
2. Runs XFakeNetPipeline on 5 LIAR statements (text only) and 6 FakeNewsNet items
   (headline + image; 3 above and 3 below the C-score threshold).

Output: runs/modality.joblib, results/week4/pipeline_demo.json, results/week4/pipeline_demo.md
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import load_config, p, set_seed, write_json  # noqa: E402


def fit_modality(cfg: dict, df: pd.DataFrame, T: np.ndarray, I: np.ndarray) -> dict:
    import joblib
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import StratifiedKFold
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    y = (df["fnn_label"] == "fake").astype(int).to_numpy()
    C = cfg["modality"]["C"]
    mk = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))  # noqa: E731
    # out-of-fold logits for the fusion weights (as in script 14), final models on all data
    folds = list(StratifiedKFold(5, shuffle=True, random_state=cfg["seed"]).split(T, df["site"] + df["fnn_label"]))
    zt, zi = np.zeros(len(y)), np.zeros(len(y))
    for tr, te in folds:
        zt[te] = mk().fit(T[tr], y[tr]).decision_function(T[te])
        zi[te] = mk().fit(I[tr], y[tr]).decision_function(I[te])
    sc = StandardScaler().fit(np.column_stack([zt, zi]))
    fus = LogisticRegression(max_iter=5000).fit(sc.transform(np.column_stack([zt, zi])), y)
    art = {"text": mk().fit(T, y), "image": mk().fit(I, y), "fusion_scaler": sc, "fusion_coef": fus.coef_[0].tolist()}
    joblib.dump(art, p(cfg["pipeline"]["modality_model"]))
    return art


def main() -> None:
    from xfakenet.clip_utils import Clip
    from xfakenet.pipeline import XFakeNetPipeline
    cfg = load_config()
    set_seed(cfg["seed"])
    out_dir = cfg["week4"]["out_dir"]

    df = pd.read_csv(p(f"{cfg['paths']['processed']}/fnn_pairs.csv"))
    df = df[df["image_path"].map(lambda x: p(x).exists())].reset_index(drop=True)
    T, I = Clip(cfg["cscore"]["clip_model"]).embed(df["title"].tolist(), [p(x) for x in df["image_path"]])
    fit_modality(cfg, df, T, I)
    print("[pipeline] modality models saved", flush=True)

    pipe = XFakeNetPipeline.load()
    test = pd.read_csv(p(f"{cfg['paths']['processed']}/corpus_test.csv"))
    rows = []
    sample = pd.concat([g.sample(2, random_state=cfg["seed"]) for _, g in test.groupby("label")]).head(5)
    for r in sample.itertuples(index=False):
        o = pipe(r.text)
        rows.append({"input": "text", "text": r.text, "gold": r.label, **{k: o[k] for k in
                     ("label", "probs", "highlighted", "top_segment", "modality_flag", "cscore",
                      "deception_pattern", "attention_highlighted")}})
    c = 1.0 - (T * I).sum(1)
    above = np.where(c > pipe.thr)[0][:3]
    below = np.where(c <= pipe.thr)[0][:3]
    for k in list(above) + list(below):
        r = df.iloc[k]
        o = pipe(r["title"], image=str(p(r["image_path"])))
        rows.append({"input": "text+image", "text": r["title"], "gold": f"FakeNewsNet {r['fnn_label']}",
                     "image": r["image_path"], **{k2: o[k2] for k2 in
                     ("label", "probs", "highlighted", "top_segment", "modality_flag", "cscore",
                      "deception_pattern")}})
    write_json(rows, f"{out_dir}/pipeline_demo.json")
    L = ["| Input | Text | Gold | Predicted | Highlighted | Modality | C-score | Pattern |",
         "|---|---|---|---|---|---|---|---|"]
    for r in rows:
        t = r["text"] if len(r["text"]) < 90 else r["text"][:87] + "…"
        L.append(f"| {r['input']} | {t} | {r['gold']} | {r['label']} | {', '.join(r['highlighted'][:4])} | "
                 f"{r['modality_flag']} | {'' if r['cscore'] is None else r['cscore']} | "
                 f"{(r['deception_pattern'] or '').replace('_', ' ')} |")
    open(p(f"{out_dir}/pipeline_demo.md"), "w").write("\n".join(L))
    print("\n".join(L))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/18_final_tables.py
"""Week 4: final results tables (classification, robustness, ablations, C-score, faithfulness, attribution).

Seeds: runs trained with seed 42 (weeks 1-3) plus any extra-seed runs named <run>_s<seed>
(e.g. xfakenet_v2_s1). With 2+ seeds, tables show mean ± standard deviation.

Output: results/final/results_tables.md, results/final/results_tables.json
"""
from __future__ import annotations

import glob
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))
from xfakenet.common import load_config, p, write_json  # noqa: E402

FAMILIES = {  # name: (metrics of seed-42 run, glob for extra seeds, fresh-attack files)
    "Baseline (RoBERTa)": ("runs/baseline/metrics.json", "runs/baseline_s*/metrics.json",
                           ["data/augment/textfooler_test.jsonl", "data/augment/textfooler_test_baseline_s*.jsonl"]),
    "XFakeNet, no augmentation": ("runs/xfakenet_noaug/metrics.json", "runs/xfakenet_noaug_s*/metrics.json",
                                  ["data/augment/textfooler_test_xfakenet_noaug.jsonl",
                                   "data/augment/textfooler_test_xfakenet_noaug_s*.jsonl"]),
    "XFakeNet v2 (augmented)": ("runs/xfakenet_v2/metrics.json", "runs/xfakenet_v2_s*/metrics.json",
                                ["data/augment/textfooler_test_xfakenet_v2.jsonl",
                                 "data/augment/textfooler_test_xfakenet_v2_s*.jsonl"]),
    "XFakeNet v3 (2nd adversarial round)": ("runs/xfakenet_v3/metrics.json", "runs/xfakenet_v3_s*/metrics.json",
                                            ["data/augment/textfooler_test_xfakenet_v3.jsonl"]),
}


def jload(f):
    f = p(f)
    return json.load(open(f)) if f.exists() else None


def ms(xs: list[float], d: int = 3) -> str:
    xs = [x for x in xs if x is not None]
    if not xs:
        return "–"
    if len(xs) == 1:
        return f"{xs[0]:.{d}f}"
    return f"{np.mean(xs):.{d}f} ± {np.std(xs, ddof=1):.{d}f}"


def attack_stats(f: str) -> dict | None:
    f = p(f)
    if not f.exists():
        return None
    t = pd.DataFrame([json.loads(l) for l in open(f)])
    vc = t["status"].value_counts()
    s, fl, n = int(vc.get("success", 0)), int(vc.get("failed", 0)), len(t)
    return {"acc_under_attack": fl / n, "success_rate": s / max(1, s + fl)}


def main() -> None:
    cfg = load_config()
    out = {}
    L = ["# XFakeNet – final results tables", ""]

    # ---- T1 classification + T2 robustness
    t1 = ["## Table 1. Classification on the clean LIAR test set (1,283 statements, 3 classes)", "",
          "| Model | Seeds | Accuracy | Macro-F1 | F1 Fabricated | F1 Misleading | F1 Credible |",
          "|---|---|---|---|---|---|---|"]
    t2 = ["## Table 2. Robustness (each model attacked by its own TextFooler run, 499 test statements)", "",
          "| Model | Seeds | Accuracy under attack | Attack success rate | Paraphrased: prediction unchanged |",
          "|---|---|---|---|---|"]
    rob = jload(f"{cfg['week3']['out_dir']}/robustness.json") or {}
    for name, (m42, extra, atk) in FAMILIES.items():
        runs = [m for m in [jload(m42)] + [jload(f) for f in sorted(glob.glob(str(p(extra))))] if m]
        if not runs:
            continue
        get = lambda k, c=None: [r["test"]["per_class"][c]["f1-score"] if c else r["test"][k] for r in runs]  # noqa
        t1.append(f"| {name} | {len(runs)} | {ms(get('accuracy'))} | {ms(get('macro_f1'))} | "
                  f"{ms(get(None, 'Fabricated'))} | {ms(get(None, 'Misleading'))} | {ms(get(None, 'Credible'))} |")
        atks = [a for f in atk for g in sorted(glob.glob(str(p(f)))) if (a := attack_stats(g))]
        para = rob.get(name, {}).get("paraphrase", {}).get("prediction_unchanged")
        if atks:
            t2.append(f"| {name} | {len(atks)} | {ms([a['acc_under_attack'] for a in atks])} | "
                      f"{ms([a['success_rate'] for a in atks])} | {'' if para is None else f'{para*100:.0f}%'} |")
        out[name] = {"n_seeds": len(runs), "macro_f1": get("macro_f1"), "accuracy": get("accuracy"),
                     "acc_under_attack": [a["acc_under_attack"] for a in atks],
                     "attack_success": [a["success_rate"] for a in atks]}
    L += t1 + ["", "*Mean ± standard deviation over seeds when more than one seed was run.*", ""] + t2 + [""]

    # ---- T3 ablations
    L += ["## Table 3. Ablations", ""]
    a, b = out.get("XFakeNet, no augmentation"), out.get("XFakeNet v2 (augmented)")
    if a and b:
        L += ["**Adversarial + paraphrase augmentation** (XFakeNet without vs with):", "",
              "| | Without | With |", "|---|---|---|",
              f"| Clean macro-F1 | {ms(a['macro_f1'])} | {ms(b['macro_f1'])} |",
              f"| Accuracy under attack | {ms(a['acc_under_attack'])} | {ms(b['acc_under_attack'])} |",
              f"| Attack success rate | {ms(a['attack_success'])} | {ms(b['attack_success'])} |", ""]
    mres = jload(f"{cfg['week3']['out_dir']}/modality_results.json")
    if mres:
        au = mres["fake_vs_real_auc_5fold"]
        L += ["**C-score signal** (FakeNewsNet fake vs real, 5-fold ROC-AUC):", "",
              "| Features | AUC |", "|---|---|"] + [f"| {k} | {v:.3f} |" for k, v in au.items()] + [""]

    # ---- T4 C-score calibration
    c = jload(f"{cfg['cscore']['out_dir']}/cscore_results.json")
    if c:
        L += ["## Table 4. C-score calibration (aligned vs mismatched pairs, test split)", "",
              f"ROC-AUC {c['test_auc_aligned_vs_mismatched']:.3f}", "",
              "| Threshold | Value | Accuracy | Flags genuine pairs | Flags mismatched pairs |", "|---|---|---|---|---|"]
        for k, r in c["thresholds"].items():
            L.append(f"| {k} | {r['threshold']:.3f} | {r['accuracy']:.3f} | {r['flag_rate_aligned']*100:.0f}% | "
                     f"{r['flag_rate_mismatched']*100:.0f}% |")
        L.append("")

    # ---- T5 faithfulness
    fm = p(f"{cfg['week4']['out_dir']}/faithfulness.md")
    if fm.exists():
        L += ["## Table 5. Explanation faithfulness (deletion test) and agreement with SHAP", "",
              fm.read_text(), ""]

    fe = p(f"{cfg['week4']['out_dir']}/faithful_explanations.md")
    if fe.exists():
        L += ["## Table 5b. Faithful explanations: attention vs occlusion vs SHAP", "", fe.read_text(), ""]

    # ---- T6 deception attribution
    d = jload(f"{cfg['week4']['out_dir']}/deception_results.json")
    if d:
        L += ["## Table 6. Deception-pattern attribution", "",
              f"- Gemini labels: {d.get('n_labelled')} pairs; categories: {d.get('label_counts')}"]
        if "classifier_cv" in d:
            cv = d["classifier_cv"]
            L.append(f"- Classifier (CLIP features, {cv['folds']}-fold CV vs Gemini labels, {cv['n']} pairs): "
                     f"accuracy {cv['accuracy']:.3f}, macro-F1 {cv['macro_f1']:.3f} "
                     f"(majority-class accuracy {cv['majority_class_accuracy']:.3f})")
        if "handcheck" in d:
            h = d["handcheck"]
            L.append(f"- Hand-check of {h['n_checked']} pairs: Gemini correct {h['gemini_accuracy']*100:.0f}%, "
                     f"Cohen's kappa {h['gemini_kappa']}"
                     + (f"; classifier agrees with human labels {h['classifier_accuracy_vs_human']*100:.0f}% "
                        f"(n={h['classifier_n']})" if "classifier_accuracy_vs_human" in h else ""))
        else:
            L.append("- Hand-check: not uploaded yet")
        L.append("")

    md = "\n".join(L)
    open(p("results/final/results_tables.md"), "w").write(md)
    write_json(out, "results/final/results_tables.json")
    print(md)


if __name__ == "__main__":
    main()

## 3. Install packages

In [ ]:
!pip install -q google-genai sentence-transformers nltk pyyaml
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| CUDA", torch.cuda.is_available())

## 0. Rebuild Week 3 outputs (only runs if they are missing from Drive)

In [ ]:
import os
if REBUILD_W3:
    if not os.path.exists("data/processed/corpus_aug_train.csv"):
        !python scripts/10_build_augmented.py
    for data, run in [("orig", "xfakenet_noaug"), ("aug", "xfakenet_v2")]:
        if not os.path.exists(f"runs/{run}/xfakenet.pt"):
            !python scripts/11_train_xfakenet.py --data {data} --run-name {run} 2>&1 | grep --line-buffered -E "^\[xfakenet.*(epoch .* done|test:)|Error|Traceback"
        !python scripts/07_textfooler_generate.py --split test --victim runs/{run} --victim-kind xfakenet --out data/augment/textfooler_test_{run}.jsonl 2>&1 | grep -E "^\[textfooler\] finished|Error|Traceback"
    !python scripts/12_evaluate_robustness.py 2>&1 | grep -v "^\[robust\]" | grep -v Warning
    !python scripts/14_modality_flag.py 2>&1 | grep -E "flag_counts|Error|Traceback"
    print("Week 3 outputs rebuilt (they are saved to Drive at the end of the notebook).")
else:
    print("Week 3 outputs found; nothing to rebuild.")

## A. Faithfulness test and SHAP agreement

In [ ]:
import os
if not os.path.exists("results/week4/faithfulness.json"):
    !python scripts/15_faithfulness.py 2>&1 | grep -v "^\[faith\]" | grep -v Warning
else:
    print(open("results/week4/faithfulness.md").read())

## B. Six deception patterns (Gemini labels → classifier → hand-check page)

In [ ]:
import os
key = os.environ.get("GEMINI_API_KEY")
if not key:
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
    except Exception as e:
        print("No Colab secret found:", type(e).__name__)
if not key:
    import getpass
    key = getpass.getpass("Paste your Gemini API key (or press Enter to skip labelling): ").strip()
if key:
    os.environ["GEMINI_API_KEY"] = key
!python scripts/16_deception.py --stage label 2>&1 | grep --line-buffered -E "^\[deception\]|Error|Traceback"
!python scripts/16_deception.py --stage train 2>&1 | grep -E "^\[deception\]|Error|Traceback"
!python scripts/16_deception.py --stage page 2>&1 | grep -E "^\[deception\]|Error|Traceback"
if os.path.exists("results/week4/handcheck.html"):
    from google.colab import files
    files.download("results/week4/handcheck.html")
    print("handcheck.html is downloading: open it in Safari and start checking while the rest runs.")

## C. Integrated pipeline on example inputs

In [ ]:
!python scripts/17_pipeline_demo.py 2>&1 | grep -v "^\[pipeline\]" | grep -v Warning

## D. Three random seeds for the main models
Seed 42 runs already exist from Weeks 1–3; this adds seeds 1 and 2 and attacks each model with TextFooler. The extra models are kept on Colab's local disk (not your Drive); only their results are saved to Drive.

In [ ]:
import os, shutil, subprocess

def lines(f):
    return sum(1 for _ in open(f)) if os.path.exists(f) else 0

def sh(cmd, pattern):
    r = subprocess.run(cmd + f' 2>&1 | grep -E "{pattern}|Error|Traceback"', shell=True, capture_output=True, text=True)
    print(r.stdout[-1200:])

LOCAL = "/content/seed_runs"
for seed in [1, 2]:
    for fam, kind, trainer in [("baseline", "hf", "06_train_baseline.py"),
                               ("xfakenet_noaug", "xfakenet", "11_train_xfakenet.py --data orig"),
                               ("xfakenet_v2", "xfakenet", "11_train_xfakenet.py --data aug")]:
        tag = f"{fam}_s{seed}"
        local, drive_dir = f"{LOCAL}/{tag}", f"runs/{tag}"
        victim = f"{local}/best" if kind == "hf" else local
        attack_file = f"data/augment/textfooler_test_{tag}.jsonl"
        trained = os.path.exists(f"{drive_dir}/metrics.json")
        attacked = lines(attack_file) >= 499
        if trained and attacked:
            print(f"{tag}: already done"); continue
        if not os.path.exists(victim):        # (re)train on local disk
            print(f"--- training {tag}")
            sh(f"python scripts/{trainer} --seed {seed} --run-name {tag} --out-dir {local}", r"^\[(baseline|xfakenet)")
            shutil.rmtree(f"{local}/ckpt", ignore_errors=True)
            os.makedirs(drive_dir, exist_ok=True)
            for fn in ["metrics.json", "confusion_test.csv"]:
                if os.path.exists(f"{local}/{fn}"): shutil.copy(f"{local}/{fn}", f"{drive_dir}/{fn}")
        if not attacked and os.path.exists(victim):
            print(f"--- attacking {tag}")
            sh(f"python scripts/07_textfooler_generate.py --split test --victim {victim} --victim-kind {kind} --out {attack_file}",
               r"^\[textfooler\] (finished|victim)")
        print(f"{tag}: trained={os.path.exists(drive_dir + '/metrics.json')} attacked={lines(attack_file)}/499")

## E. Optional: second round of adversarial training
Only worth running if the seed results show augmentation helps but the model is still easy to fool. Set `ROUND2 = True` and run this cell.

In [ ]:
ROUND2 = False
if ROUND2:
    import os
    if not os.path.exists("runs/xfakenet_v3/metrics.json"):
        !python scripts/07_textfooler_generate.py --split train --victim runs/xfakenet_v2 --victim-kind xfakenet --out data/augment/textfooler_train_r2.jsonl 2>&1 | grep --line-buffered -E "^\[textfooler\]|Error|Traceback"
        !python scripts/10_build_augmented.py --extra-tf data/augment/textfooler_train_r2.jsonl --out data/processed/corpus_aug2_train.csv
        !python scripts/11_train_xfakenet.py --data aug --train-file data/processed/corpus_aug2_train.csv --run-name xfakenet_v3 2>&1 | grep --line-buffered -E "^\[xfakenet|Error|Traceback"
    !python scripts/07_textfooler_generate.py --split test --victim runs/xfakenet_v3 --victim-kind xfakenet --out data/augment/textfooler_test_xfakenet_v3.jsonl 2>&1 | grep -E "^\[textfooler\] finished|Error|Traceback"
else:
    print("Round 2 is off (ROUND2 = False).")

## F. Final results tables

In [ ]:
!python scripts/16_deception.py --stage score 2>&1 | grep -E "^\[deception\]|Error"
!python scripts/18_final_tables.py > /dev/null 2>&1
from IPython.display import Markdown, display
display(Markdown(open("results/final/results_tables.md").read()))

import os, shutil
os.makedirs("/content/w4", exist_ok=True)
for src in ["results/final/results_tables.md", "results/final/results_tables.json",
            "results/week4/faithfulness.md", "results/week4/faithfulness.json",
            "results/week4/deception_results.json", "results/week4/handcheck.html",
            "results/week4/pipeline_demo.md", "results/week4/pipeline_demo.json",
            "data/deception/gemini_labels.jsonl"]:
    if os.path.exists(src): shutil.copy(src, "/content/w4/" + os.path.basename(src))
shutil.make_archive("/content/xfakenet_week4_results", "zip", "/content/w4")
from google.colab import files
files.download("/content/xfakenet_week4_results.zip")

from google.colab import drive
drive.flush_and_unmount()     # make sure every file has finished uploading to Drive
drive.mount("/content/drive"); os.chdir(WORK)
print("All results saved to Google Drive.")

## G. Upload your hand-check answers
When you've finished the hand-check page, click **Download results (CSV)** there, then set the line below to `True`, run this cell, and choose `handcheck_results.csv`. It updates the tables and downloads a new zip.

In [ ]:
I_HAVE_FINISHED_THE_HANDCHECK = False
if I_HAVE_FINISHED_THE_HANDCHECK:
    from google.colab import files
    import shutil, os
    up = files.upload()
    name = next(n for n in up if n.endswith(".csv"))
    shutil.move(name, "data/deception/handcheck_results.csv")
    !python scripts/16_deception.py --stage score
    !python scripts/18_final_tables.py > /dev/null 2>&1
    for src in ["results/final/results_tables.md", "results/week4/deception_results.json"]:
        shutil.copy(src, "/content/w4/" + os.path.basename(src))
    shutil.copy("data/deception/handcheck_results.csv", "/content/w4/handcheck_results.csv")
    shutil.make_archive("/content/xfakenet_week4_results", "zip", "/content/w4")
    files.download("/content/xfakenet_week4_results.zip")
else:
    print("Set I_HAVE_FINISHED_THE_HANDCHECK = True after downloading your answers from the hand-check page.")